# V10 — DINOv3 ViT-7B (timm) beku + *rich spatial readout*, satu notebook end-to-end

**BDC Satria Data 2026 — klasifikasi citra sampah (0 = Recyclable, 1 = Electronic, 2 = Organic), metrik Macro F1.**

Notebook ini adalah satu alur penuh: audit environment → cache fitur backbone beku → seleksi head *train-only* → retraining final → inferensi test → submission → artefak reproduksibilitas.

## Ringkasan eksekutif eksperimen

| Aspek | Isi |
|---|---|
| Backbone | `timm/vit_7b_patch16_dinov3.lvd1689m` (DINOv3 ViT-7B, LVD-1689M), **100% beku**, FP16, di-*shard* ke dua Tesla T4 |
| Input | 256×256, satu tampilan asli, resize kotak langsung, bilinear + antialias, normalisasi ImageNet |
| Representasi baru | **17 cabang** per gambar: 8 cabang CLS/patch-mean lintas 4 blok terakhir + 9 cabang spasial/distribusional dari blok terakhir |
| Kandidat | Tepat dua: `timm_control` (padanan V7) dan `rich_spatial_fusion` |
| Seleksi | Bertahap dan terkunci: fold 0 → gerbang → fold 1–4 → `selection_decision.json` |
| Model final | **Satu** head tunggal, dilatih ulang dari nol pada seluruh 26.527 baris train resmi |
| Inferensi | Satu forward pass deterministik, argmax, tanpa TTA / ensemble / blending |

> **Status hasil.** Semua angka V10 di notebook ini *belum ada* sampai Anda menjalankannya di Kaggle. Angka V7/V9A yang dikutip adalah **hasil historis proyek** dari `results/`, bukan klaim tentang V10. Notebook ini tidak mengklaim V10 lebih baik daripada V7 sebelum dijalankan.


## 1. Batasan kompetisi dan kebijakan anti-kebocoran

Aturan resmi (Petunjuk Teknis BDC Satria Data 2026, Bagian 5 dan 7) yang ditegakkan secara programatik di notebook ini:

1. **Hanya citra resmi panitia.** Tidak ada citra berlabel dari sumber luar.
2. **Label kanonis** berasal dari folder kelas resmi `0_Recyclable`, `1_Electronic`, `2_Organic`. Tidak ada baris train yang dibuang atau di-*relabel*.
3. **Hanya informasi visual.** Nama file, path, ID, ukuran file, dimensi citra, dan EXIF **tidak pernah** menjadi fitur prediktif. Dimensi citra hanya dipakai sebagai penjaga memori dekoder.
4. **Data Uji hanya untuk prediksi akhir.** Test tidak dipakai untuk training, validasi, seleksi model, kalibrasi, penyetelan ambang, maupun *hyperparameter tuning*.
5. **Tidak ada pseudo-labeling test**, tidak ada inspeksi manual test untuk keputusan model, tidak ada label test eksternal, tidak ada pembacaan submission sebelumnya untuk blending atau koreksi.
6. **Tidak ada umpan balik leaderboard** yang masuk ke pipeline.
7. **Isolasi test bersifat mekanis.** Direktori test dan template resmi baru di-*resolve* setelah `selection_decision.json` tertulis dan tanda tangannya cocok. Sebelum titik itu, pencarian direktori train secara eksplisit memangkas setiap folder bernama `test` tanpa mengenumerasinya.
8. **Urutan baris submission** persis mengikuti template resmi; keluaran berupa kode numerik 0, 1, 2 pada kolom `id,predicted`.

Bendera `test_images_accessed` dan `manual_test_labels_accessed` dicatat pada setiap artefak; keduanya wajib `false` selama seluruh fase seleksi.

### Kontrak model tunggal

Prediksi akhir berasal dari **satu** backbone beku + **satu** head terpilih + **satu** tampilan citra + **satu** forward pass + **satu** checkpoint hasil full-training. Dilarang: ensemble antar-fold, ensemble antar-seed, blending probabilitas, *model soup*, head spesialis, blending V7/V10, blending submission lama, TTA, multi-crop, dan inferensi *horizontal flip*. Model lintas-fold hanya dilatih sementara untuk bukti *train-only* dan tidak pernah digabungkan.


## 2. Mengapa V7 dan V9A mentok

**V7 (diadopsi).** DINOv3 ViT-7B beku, 256 px, LayerNorm final resmi per token sebelum pooling, cache empat blok terakhir berisi **hanya** CLS dan rata-rata 256 patch token. Head `regularized_mlp_final`. OOF lima fold Macro F1 = `0.9901360067264733`, total 287 error, dengan 241 di antaranya Recyclable ↔ Organic. Electronic praktis jenuh.

**V8 (ditolak).** Spesialis biner Recyclable/Organic justru menambah error pada fold 0 (61 → 64/65) karena kehilangan konteks batas tiga kelas gabungan.

**V9 (ditolak).** *Drop* dan *relabel* terhadap mislabel yang jelas menurunkan Macro F1 fold 0 (0.989918 → 0.989471 / 0.989303). Raw train original tetap kanonis.

**V9A (ditolak untuk adopsi).** Ini yang paling menentukan arah V10. V9A menjalankan *head zoo* besar di atas cache V7 yang sama: 12 readout × ~10 keluarga model + blend berpasangan + spesialis R/O. Bukti dari `results/v9a_v7_cached_head_zoo/`:

| Readout (skrining fold 0) | Dim | Macro F1 |
|---|---|---|
| `mean_last4_concat` | 8192 | 0.98689 |
| `layer_mean_std_concat` | 16384 | 0.98662 |
| `final_concat` (readout V7) | 8192 | 0.98587 |
| `final_cls__l2` | 4096 | 0.98028 |
| `final_patch__l2` | 4096 | 0.97469 |

Pada fold 0, readout lintas-lapisan tampak unggul. Namun pada fold penilaian 1–4 keunggulan itu **tidak bertahan**: kontrol `torch_mlp512 + final_concat` justru menang agregat (`0.9903008`, 225 error) dibanding `torch_msd_mlp512 + mean_last4_concat` (`0.9903328`, **227** error), dan resep terbaik keseluruhan hanyalah sebuah *blend* dengan gain `+0.0001385`, menang di 2 dari 4 fold. `final_train_only_decision.json` mencatat `adoption_recommendation: false`.

**Kesimpulan yang dipakai V10.** Dua hal sudah terbukti habis di atas cache V7: (a) variasi arsitektur head, dan (b) **keragaman lintas-lapisan** dari blok 36–39. Yang belum pernah diuji adalah struktur **di dalam** citra, karena cache V7 merata-ratakan seluruh 256 patch token menjadi satu vektor dan membuang informasi itu secara permanen.


## 3. Hipotesis V10, alternatif yang dipertimbangkan, dan risiko

### Pendekatan yang dipilih

Bangun **cache embedding baru yang sadar-patch** dari DINOv3 ViT-7B beku (implementasi timm), lalu bandingkan tepat dua head dengan protokol pelatihan identik:

* `timm_control` — CLS blok terakhir ⊕ rata-rata 256 patch token asli (8192-d). Ini padanan V7 di atas cache baru, sekaligus penanda apakah backend timm mengubah representasi.
* `rich_spatial_fusion` — LayerNorm per cabang (17×), lalu satu proyeksi `Linear(4096, 256)` yang **dibagi** antar cabang, GELU, dropout 0.10, konkatenasi 17×256 = 4352, lalu `LayerNorm → Linear(512) → GELU → Dropout(0.40) → Linear(3)`.

Proyeksi dibagi karena seluruh cabang hidup di ruang fitur DINO yang sama; LayerNorm per cabang tetap memberi setiap cabang skala sendiri.

### Informasi yang benar-benar baru dibanding V7 dan V9A

Sembilan cabang blok-terakhir tidak dapat direkonstruksi dari cache V7 mana pun:

* **`final_patch_std`** — sebaran/tekstur antar-patch (momen kedua).
* **`final_patch_max`** — respons ekstrem per dimensi fitur; "adakah satu patch pun yang sangat kuat pada fitur ini".
* **`final_top10/25/50_salient_mean`** — rata-rata patch paling saliens, diperingkat oleh **kosinus CLS↔patch**, bukan norma L2 token (setelah LayerNorm norma token nyaris seragam sehingga tidak informatif untuk peringkat).
* **4 cabang kuadran** — tata letak spasial kasar: apakah materialnya menempati objek utama atau hanya latar.

Justru sinyal seperti inilah yang secara apriori relevan untuk Recyclable vs Organic (kulit buah vs kemasan cokelat, sisa makanan di dalam wadah plastik), yaitu 84% dari sisa error V7.

### Alternatif yang dipertimbangkan dan alasan penolakan

| Alternatif | Alasan tidak dipilih |
|---|---|
| *Head zoo* baru di atas cache V7 | Sudah dijalankan sebagai V9A; gain terbaik `+0.00014`, tidak stabil, `adoption_recommendation: false`. Diminishing returns terbukti. |
| Menjadikan readout lintas-lapisan sebagai sinyal utama | V9A sudah menguji `mean_last4_concat` dan `layer_mean_std_concat`; keunggulan fold 0 tidak tereplikasi pada fold 1–4. Tetap disimpan sebagai 8 cabang, tetapi bukan sumber harapan. |
| Ganti backend Transformers → timm saja | Model card hanya menyebut QKV bias nol yang tidak dimuat dan periode RoPE float32; keduanya tidak menjanjikan gain besar. Backend bukan hipotesis, hanya jalur implementasi untuk readout yang lebih kaya. |
| Resolusi lebih tinggi (mis. 512 px → 1024 patch) | Biaya ekstraksi naik sekitar 4× (>7 jam), melewati tenggat, sekaligus mencampur perubahan geometri dengan perubahan representasi. |
| *Multi-view readout* ala DINO-MVR (`ref/papers/2605.07221v1`) | Tuas terkuat pada paper itu adalah agregasi banyak tampilan saat inferensi — dilarang keras oleh kontrak model tunggal. |
| Unfreeze parsial / LoRA pada ViT-7B | Tidak layak di T4×2 dan sudah ditutup pada handoff proyek. |
| Data cleaning, relabel, spesialis R/O | Sudah ditolak sebagai V9 dan V8. |

Tidak ada alternatif yang secara jelas mendominasi *patch-aware pooling* dengan syarat: berjalan di T4×2, *train-only*, tanpa data luar, backbone beku, satu notebook, satu model final, tanpa *head zoo* baru, dan realistis dikerjakan hari ini. Karena itu desain sadar-patch dipakai.

### Keputusan tentang 17 cabang

Rancangan 17 cabang **dipertahankan utuh**. Pertimbangan membuang `final_patch_max` sebagai cabang paling berisik ditinjau dan ditolak: (a) V9A menunjukkan yang benar-benar merusak adalah kombinasi elementwise non-linear (`final_absdiff__l2` 0.9415, `final_product__l2` 0.9384), sedangkan statistik distribusional bertingkah wajar (`layer_mean_std_concat` setara `mean_last4_concat`); (b) setiap cabang punya LayerNorm sendiri sehingga skalanya dinormalkan sebelum masuk proyeksi bersama; (c) membuang cabang tanpa bukti hanya akan menjadi tebakan, dan pencarian kombinatorial antar-cabang dilarang secara eksplisit. Delapan cabang lintas-lapisan tetap disimpan karena hampir gratis (proyeksi dibagi) dan menjaga cache tetap superset dari readout V7 dan V9A.

### Risiko yang tersisa

1. Cabang kuadran dan `max` dapat menyerap isyarat latar belakang dan justru menambah *overfitting*.
2. Fusi 4352-d di atas ~21 ribu baris latih bisa terlalu besar meskipun sudah diregularisasi.
3. Sebagian error R↔O kemungkinan adalah **ambiguitas label** (isi vs kemasan) yang tidak dapat diselesaikan readout beku mana pun — V9 sudah memperlihatkan gejalanya.
4. Backend timm belum pernah dijalankan pada proyek ini; padanan numerik terhadap V7 hanya bisa diverifikasi saat runtime.
5. Gerbang `+0.0015` sengaja lebih besar daripada simpangan antar-fold V9A (`std ≈ 0.0008`). Konsekuensinya penolakan adalah hasil yang wajar, dan notebook tetap menghasilkan submission kontrol.


## 4. Instruksi runtime (wajib dibaca sebelum Run All)

Notebook ini **hanya** dirancang untuk Kaggle.

1. **Accelerator:** `GPU T4 x2`. Notebook gagal-tertutup jika tidak melihat tepat dua GPU dan keduanya bukan Tesla T4.
2. **Internet:** ON. Checkpoint diunduh dari Hugging Face (± 26,9 GB, satu berkas `model.safetensors` FP32).
3. **Kaggle Secret:** buat secret bernama `HF_TOKEN` (Add-ons → Secrets) dan pastikan lisensi DINOv3 sudah disetujui pada akun HF yang sama.
4. **Dataset input:** lampirkan dataset kompetisi sehingga terdapat satu direktori `train` berisi persis `0_Recyclable` (9.999), `1_Electronic` (3.961), `2_Organic` (12.567), plus direktori `test` (1.458 citra) dan template submission resmi.
5. **Isi `TEAM_NAME`** pada sel konfigurasi. Bila dibiarkan `CHANGE_ME`, notebook tetap jalan tetapi mencetak peringatan besar dan nama berkas submission ikut salah.
6. **Perkiraan waktu:** unduh + muat model 10–25 menit, ekstraksi fitur train ± 1–2 jam, ekstraksi test ± 5 menit, seluruh pelatihan head < 15 menit. Total wajar 2–3 jam dan muat dalam satu sesi Kaggle.
7. **Jika sesi terputus:** jalankan ulang dari atas. Shard cache yang sudah valid akan dipakai kembali, bukan dihitung ulang. Untuk melanjutkan lintas-sesi, simpan output run sebelumnya sebagai Kaggle Dataset dan lampirkan; shard hanya akan dipakai bila tanda tangan cache-nya identik.

**Melanjutkan lintas-sesi.** Shard dicari di `/kaggle/working/<nama eksperimen>/feature_cache/...` lalu di setiap akar pada `PRIOR_CACHE_SEARCH_ROOTS`. Validasi memakai metadata lengkap termasuk tanda tangan cache, sehingga cache V7 lama, cache dengan definisi cabang berbeda, atau shard rusak **tidak mungkin** terpakai secara diam-diam.

**Catatan implementasi timm yang dicatat di manifest.** (a) Bobot asli memiliki seluruh bias proyeksi QKV bernilai nol, sehingga timm menonaktifkan `qkv_bias` alih-alih memuat parameter nol. (b) Model asli menyimpan periode RoPE melalui bfloat16, sedangkan timm menginisialisasinya di float32; perilaku default float32 timm **dipertahankan** karena tidak ada bukti berbasis kode yang menuntut perubahan. Kedua hal ini adalah perbedaan implementasi, bukan sumber gain yang diharapkan.


## 5. Dependensi Kaggle

PyTorch, torchvision, pandas, scikit-learn, dan matplotlib disediakan Kaggle dan tidak diganti. Yang dipasang hanya paket untuk memuat dan men-*dispatch* checkpoint besar: `timm` (arsitektur DINOv3 ViT-7B), `accelerate` (penempatan multi-GPU), `safetensors`, dan `huggingface_hub`. `transformers` ikut dipasang karena dipakai oleh jalur cadangan yang dinyatakan eksplisit pada konfigurasi (default tidak aktif).


In [ ]:
%pip install -q "timm>=1.0.20" "accelerate>=1.10,<2" "safetensors>=0.4" "huggingface_hub>=0.34" "transformers>=4.57,<5"

## 6. Import dan konfigurasi global

Seluruh keputusan ilmiah didefinisikan sekali di sini, lalu dikunci dengan `assert` dan sebuah tanda tangan SHA-256. Tidak ada konstanta ajaib yang tersebar di sel lain.

Urutan 17 cabang bersifat **deterministik** dan menjadi bagian dari identitas cache: delapan cabang lintas-lapisan (blok ke-4-terakhir sampai blok terakhir, masing-masing CLS lalu rata-rata patch), disusul sembilan cabang spasial/distribusional dari blok terakhir. Indeks cabang untuk head kontrol diturunkan dari nama, bukan di-*hardcode*.


In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
import zipfile
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode, v2
from tqdm.auto import tqdm

NOTEBOOK_STARTED_AT = time.perf_counter()

# ---------------------------------------------------------------------------
# USER CONFIGURATION
# ---------------------------------------------------------------------------
TEAM_NAME = 'CHANGE_ME'  # TODO(user): isi dengan nama tim resmi sebelum submission.
BACKBONE_BACKEND = 'timm'  # 'timm' (utama) atau 'transformers' (cadangan eksplisit, tidak pernah aktif sendiri).
GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED = True
ZIP_INCLUDE_FEATURE_CACHE = False
FEATURE_BATCH_SIZE = 4  # Turunkan ke 2 atau 1 jika preflight melaporkan kehabisan memori GPU.
FEATURE_LOADER_WORKERS = 2

# ---------------------------------------------------------------------------
# EXPERIMENT IDENTITY
# ---------------------------------------------------------------------------
EXPERIMENT_NAME = 'v10_dinov3_vit7b_timm_rich_spatial'
EXPERIMENT_PURPOSE = 'patch-aware rich spatial readout of a frozen DINOv3 ViT-7B, single-model end-to-end'
REVISION_MARKER = 'v10_timm_rich_spatial_17branch_v1'
CACHE_SCHEMA_VERSION = 1
HEAD_SCHEMA_VERSION = 1
PREFLIGHT_SCHEMA_VERSION = 1

# ---------------------------------------------------------------------------
# BACKBONE AND TOKEN CONTRACT
# ---------------------------------------------------------------------------
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
TIMM_HF_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
TRANSFORMERS_MODEL_ID = 'facebook/dinov3-vit7b16-pretrain-lvd1689m'
MODEL_REVISION_REQUEST = 'main'
IMAGE_SIZE = 256
PATCH_SIZE_EXPECTED = 16
GRID_SIZE = IMAGE_SIZE // PATCH_SIZE_EXPECTED
EXPECTED_HIDDEN_SIZE = 4096
EXPECTED_PATCH_TOKENS = GRID_SIZE * GRID_SIZE
EXPECTED_REGISTER_TOKENS = 4
EXPECTED_PREFIX_TOKENS = 1 + EXPECTED_REGISTER_TOKENS
EXPECTED_SEQUENCE_TOKENS = EXPECTED_PREFIX_TOKENS + EXPECTED_PATCH_TOKENS
LAST_N_BLOCKS = 4
FEATURE_VIEW = 'orig'
BACKBONE_FROZEN = True
BACKBONE_DTYPE = torch.float16
BACKBONE_DTYPE_NAME = 'float16'
POOLING_COMPUTE_DTYPE = torch.float32
POOLING_COMPUTE_DTYPE_NAME = 'float32'
FEATURE_STORAGE_DTYPE = torch.float16
FEATURE_STORAGE_DTYPE_NAME = 'float16'
SHARDING_STRATEGY = 'balanced'
SHARDING_MAX_MEMORY = {0: '13GiB', 1: '13GiB'}
ATTENTION_IMPLEMENTATION = 'sdpa'
FINAL_NORM_EQUIVALENCE_ATOL = 2e-3
FINAL_NORM_EQUIVALENCE_RTOL = 2e-3
POOLING_SELF_CONSISTENCY_ATOL = 1e-3
MIN_EXTRACTION_IMAGES_PER_SECOND = 0.8
BENCHMARK_WARMUP_IMAGES = 8
BENCHMARK_TIMED_IMAGES = 24
FEATURE_SHARD_SIZE = 256

# ---------------------------------------------------------------------------
# ORDERED 17-BRANCH RICH EMBEDDING CONTRACT
# ---------------------------------------------------------------------------
SALIENT_FRACTIONS = (0.10, 0.25, 0.50)
SALIENT_TOP_K = tuple(int(round(fraction * EXPECTED_PATCH_TOKENS)) for fraction in SALIENT_FRACTIONS)
CROSS_LAYER_BRANCH_NAMES = tuple(
    'block_m{}_{}'.format(offset, kind)
    for offset in range(LAST_N_BLOCKS, 0, -1)
    for kind in ('cls', 'patch_mean')
)
FINAL_SPATIAL_BRANCH_NAMES = (
    'final_patch_std',
    'final_patch_max',
    'final_top10_salient_mean',
    'final_top25_salient_mean',
    'final_top50_salient_mean',
    'final_quadrant_top_left_mean',
    'final_quadrant_top_right_mean',
    'final_quadrant_bottom_left_mean',
    'final_quadrant_bottom_right_mean',
)
BRANCH_NAMES = CROSS_LAYER_BRANCH_NAMES + FINAL_SPATIAL_BRANCH_NAMES
NUM_BRANCHES = len(BRANCH_NAMES)
CONTROL_BRANCH_NAMES = ('block_m1_cls', 'block_m1_patch_mean')
CONTROL_BRANCH_INDICES = tuple(BRANCH_NAMES.index(name) for name in CONTROL_BRANCH_NAMES)

# ---------------------------------------------------------------------------
# OFFICIAL DATA CONTRACT
# ---------------------------------------------------------------------------
CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_RAW_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_TRAIN_ROWS = 26527
EXPECTED_TEST_ROWS = 1458
SUBMISSION_COLUMNS = ['id', 'predicted']
ALLOWED_IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SOURCE_SIDE = 4096
SAFE_DECODE_MAX_SIDE = 1024
PIL_MAX_IMAGE_PIXELS = 200_000_000

# ---------------------------------------------------------------------------
# FOLD AND HEAD TRAINING PROTOCOL (identical for both candidates)
# ---------------------------------------------------------------------------
SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0
ASSESSMENT_FOLDS = (1, 2, 3, 4)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 5
HEAD_LEARNING_RATE = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH_SIZE = 256
PREDICT_BATCH_SIZE = 512
GRAD_CLIP_MAX_NORM = 1.0
LABEL_SMOOTHING = 0.05
CONTROL_HIDDEN_DIM = 512
CONTROL_DROPOUT = 0.40
RICH_PROJECTION_DIM = 256
RICH_PROJECTION_DROPOUT = 0.10
RICH_FUSION_HIDDEN_DIM = 512
RICH_FUSION_DROPOUT = 0.40
CONTROL_CANDIDATE = 'timm_control'
RICH_CANDIDATE = 'rich_spatial_fusion'
CANDIDATE_NAMES = (CONTROL_CANDIDATE, RICH_CANDIDATE)

# ---------------------------------------------------------------------------
# LOCKED SELECTION GATES (fixed before any V10 result exists)
# ---------------------------------------------------------------------------
DEV_MIN_MACRO_F1_GAIN = 0.0015
DEV_MAX_ELECTRONIC_F1_DROP = 0.0005
ASSESSMENT_MIN_MACRO_F1_GAIN = 0.0015
ASSESSMENT_MIN_FOLDS_WON = 3
ASSESSMENT_MAX_ELECTRONIC_F1_DROP = 0.0005
ASSESSMENT_MAX_SINGLE_FOLD_GAIN_SHARE = 0.75

# ---------------------------------------------------------------------------
# HISTORICAL PROJECT REFERENCES (measured facts from results/, not V10 claims)
# ---------------------------------------------------------------------------
V7_REFERENCE = {
    'status': 'historical adopted result, transformers backend, CLS + patch-mean readout',
    'model_id': TRANSFORMERS_MODEL_ID,
    'fold0_macro_f1': 0.989918,
    'five_fold_oof_macro_f1': 0.9901360067264733,
    'five_fold_oof_total_errors': 287,
    'five_fold_oof_recyclable_organic_errors': 241,
}
V9A_REFERENCE = {
    'status': 'historical rejected result from results/v9a_v7_cached_head_zoo/final_train_only_decision.json',
    'best_recipe_type': 'pairwise blend, forbidden by the V10 single-model contract',
    'best_assessment_macro_f1': 0.9904393207699855,
    'control_assessment_macro_f1': 0.9903007998213612,
    'macro_f1_gain_over_control': 0.00013852094862432107,
    'folds_won_against_control': 2,
    'assessment_fold_macro_f1_std': 0.000720562511214698,
    'adoption_recommendation': False,
}

# ---------------------------------------------------------------------------
# CONTRACT ASSERTIONS
# ---------------------------------------------------------------------------
assert BACKBONE_BACKEND in {'timm', 'transformers'}
assert IMAGE_SIZE == 256 and PATCH_SIZE_EXPECTED == 16 and GRID_SIZE == 16
assert EXPECTED_PATCH_TOKENS == 256 and EXPECTED_PREFIX_TOKENS == 5 and EXPECTED_SEQUENCE_TOKENS == 261
assert EXPECTED_PREFIX_TOKENS != EXPECTED_PATCH_TOKENS
assert SALIENT_TOP_K == (26, 64, 128)
assert NUM_BRANCHES == 17 and len(set(BRANCH_NAMES)) == NUM_BRANCHES
assert CROSS_LAYER_BRANCH_NAMES[0] == 'block_m4_cls' and CROSS_LAYER_BRANCH_NAMES[-1] == 'block_m1_patch_mean'
assert CONTROL_BRANCH_INDICES == (6, 7)
assert BACKBONE_FROZEN and BACKBONE_DTYPE is torch.float16
assert POOLING_COMPUTE_DTYPE is torch.float32 and FEATURE_STORAGE_DTYPE is torch.float16
assert SEED == 42 and N_FOLDS == 5 and DEVELOPMENT_FOLD == 0 and ASSESSMENT_FOLDS == (1, 2, 3, 4)
assert MAX_EPOCHS == 15 and EARLY_STOPPING_PATIENCE == 5
assert FEATURE_BATCH_SIZE >= 1 and FEATURE_SHARD_SIZE >= 1
assert sum(EXPECTED_RAW_COUNTS.values()) == EXPECTED_TRAIN_ROWS

# ---------------------------------------------------------------------------
# TEST-ISOLATION STATE (flipped only inside the post-lock test section)
# ---------------------------------------------------------------------------
TEST_ACCESS_STATE = {'test_images_accessed': False, 'manual_test_labels_accessed': False}

print('Experiment      :', EXPERIMENT_NAME)
print('Backbone backend:', BACKBONE_BACKEND)
print('Branch order    :')
for branch_position, branch_name in enumerate(BRANCH_NAMES):
    print('  [{:2d}] {}'.format(branch_position, branch_name))
print('Control branches:', CONTROL_BRANCH_NAMES, '->', CONTROL_BRANCH_INDICES)
if TEAM_NAME == 'CHANGE_ME':
    print('\n' + '!' * 78)
    print('!! TEAM_NAME masih "CHANGE_ME". Isi TEAM_NAME sebelum mengumpulkan submission. !!')
    print('!' * 78)

## 7. Utilitas reproduksibilitas dan penulisan artefak atomik

Seed dikunci di 42 dan algoritma deterministik diaktifkan bila tersedia. Semua JSON, CSV, NPY, dan checkpoint ditulis lewat berkas sementara lalu `os.replace`, sehingga sesi Kaggle yang terputus tidak pernah meninggalkan manifest, shard, atau checkpoint setengah jadi yang bisa lolos validasi pada run berikutnya.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')
Image.MAX_IMAGE_PIXELS = PIL_MAX_IMAGE_PIXELS
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_WORKING.is_dir():
    raise RuntimeError(
        'This notebook requires a Kaggle runtime with /kaggle/working. '
        'Run it on Kaggle with the GPU T4 x2 accelerator.'
    )
ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
CACHE_DIR = ARTIFACT_DIR / 'feature_cache'
TRAIN_CACHE_DIR = CACHE_DIR / 'train'
TEST_CACHE_DIR = CACHE_DIR / 'test'
HEADS_DIR = ARTIFACT_DIR / 'heads'
FIGURES_DIR = ARTIFACT_DIR / 'figures'
for directory in (ARTIFACT_DIR, CACHE_DIR, TRAIN_CACHE_DIR, TEST_CACHE_DIR, HEADS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

PRIOR_CACHE_SEARCH_ROOTS = (Path('/kaggle/input'),)
PREFLIGHT_REPORT_PATH = ARTIFACT_DIR / 'preflight_report.json'
SELECTION_DECISION_PATH = ARTIFACT_DIR / 'selection_decision.json'


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, default=json_default)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path, index=False):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def host_rss_gib():
    resident_pages = int(Path('/proc/self/statm').read_text().split()[1])
    return resident_pages * os.sysconf('SC_PAGE_SIZE') / 2 ** 30


def gpu_memory_rows():
    rows = []
    for device_id in range(torch.cuda.device_count()):
        rows.append({
            'device_id': device_id,
            'allocated_gib': torch.cuda.memory_allocated(device_id) / 2 ** 30,
            'reserved_gib': torch.cuda.memory_reserved(device_id) / 2 ** 30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(device_id) / 2 ** 30,
            'peak_reserved_gib': torch.cuda.max_memory_reserved(device_id) / 2 ** 30,
        })
    return rows


def release_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sanitize_for_filename(value):
    cleaned = ''.join(character if character.isalnum() or character in {'-', '_'} else '_' for character in str(value))
    return cleaned.strip('_') or 'UNSET'


print('Artifact directory:', ARTIFACT_DIR)
print('Seed locked at    :', SEED)

## 8. Audit environment, kontrak dua Tesla T4, dan penjaga label test manual

Notebook menolak berjalan bila tidak melihat **tepat dua** GPU dan keduanya bukan Tesla T4. Selain itu ada penjaga eksplisit: bila ada berkas CSV berlabel-manual test yang ikut terlampir sebagai input Kaggle, notebook berhenti — berkas seperti itu berisi label Data Uji dan penggunaannya adalah pelanggaran berat.

Sel ini juga menulis `environment.json`, `config.json`, dan `locked_contract.json` beserta tanda tangannya. `locked_contract.json` adalah acuan yang dipakai ulang oleh identitas cache, resep terkunci, dan keputusan seleksi.


In [ ]:
import accelerate
import safetensors
import timm
import timm.data  # explicit: timm/__init__.py does not import the data subpackage
import transformers

ENVIRONMENT_INFO = {
    'python_version': sys.version,
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'cudnn_version': torch.backends.cudnn.version(),
    'timm_version': timm.__version__,
    'accelerate_version': accelerate.__version__,
    'safetensors_version': safetensors.__version__,
    'transformers_version': transformers.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
}

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(
        'V10 requires exactly two visible CUDA devices, found {}. '
        'Set the Kaggle accelerator to "GPU T4 x2".'.format(
            torch.cuda.device_count() if torch.cuda.is_available() else 0
        )
    )
GPU_RUNTIME_INFO = []
for device_id in (0, 1):
    properties = torch.cuda.get_device_properties(device_id)
    upper_name = properties.name.upper()
    if 'TESLA T4' not in upper_name and upper_name.strip() != 'T4':
        raise RuntimeError(
            'cuda:{} is "{}", not an NVIDIA Tesla T4. The locked protocol requires two Tesla T4 GPUs.'.format(
                device_id, properties.name
            )
        )
    GPU_RUNTIME_INFO.append({
        'device_id': device_id,
        'name': properties.name,
        'total_vram_gib': properties.total_memory / 2 ** 30,
        'multi_processor_count': properties.multi_processor_count,
    })

KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_INPUT.is_dir():
    raise RuntimeError('Attach the competition dataset: /kaggle/input was not found.')

blocked_label_files = []
for candidate_path in KAGGLE_INPUT.rglob('*.csv'):
    normalized = candidate_path.name.lower().replace('_', '-')
    if 'manual' in normalized and ('label' in normalized or 'labell' in normalized):
        blocked_label_files.append(str(candidate_path))
if blocked_label_files:
    raise RuntimeError(
        'Competition safety guard triggered: manual test-label CSV files are attached as notebook inputs. '
        'Detach them before running V10:\n' + '\n'.join(blocked_label_files)
    )

LOCKED_CONTRACT = {
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'experiment_purpose': EXPERIMENT_PURPOSE,
    'backbone_backend': BACKBONE_BACKEND,
    'timm_model_name': TIMM_MODEL_NAME,
    'timm_hf_repo_id': TIMM_HF_REPO_ID,
    'transformers_model_id': TRANSFORMERS_MODEL_ID,
    'image_size': IMAGE_SIZE,
    'patch_size_expected': PATCH_SIZE_EXPECTED,
    'grid_size': GRID_SIZE,
    'feature_view': FEATURE_VIEW,
    'token_contract': {
        'sequence_token_count': EXPECTED_SEQUENCE_TOKENS,
        'prefix_token_count': EXPECTED_PREFIX_TOKENS,
        'register_token_count': EXPECTED_REGISTER_TOKENS,
        'patch_token_count': EXPECTED_PATCH_TOKENS,
        'hidden_size': EXPECTED_HIDDEN_SIZE,
        'cls_position': 'prefix token 0',
        'register_tokens_excluded_from_every_patch_statistic': True,
    },
    'branch_names': list(BRANCH_NAMES),
    'branch_count': NUM_BRANCHES,
    'control_branch_names': list(CONTROL_BRANCH_NAMES),
    'salient_top_k': list(SALIENT_TOP_K),
    'saliency_definition': 'cosine similarity between final normalized CLS and final normalized true patch tokens',
    'last_n_blocks': LAST_N_BLOCKS,
    'per_token_normalization': 'official pretrained DINOv3 final norm, applied exactly once per token',
    'normalization_timing': 'before CLS extraction and every patch statistic',
    'backbone_frozen': BACKBONE_FROZEN,
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'sharding_strategy': SHARDING_STRATEGY,
    'sharding_max_memory': {str(key): value for key, value in SHARDING_MAX_MEMORY.items()},
    'seed': SEED,
    'n_folds': N_FOLDS,
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'candidates': list(CANDIDATE_NAMES),
    'head_protocol': {
        'max_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE,
        'predict_batch_size': PREDICT_BATCH_SIZE,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'label_smoothing': LABEL_SMOOTHING,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'class_weights': 'total / (num_classes * class_count) from the current training partition',
        'model_selection_metric': 'validation Macro F1, log loss as secondary diagnostic',
        'threshold_tuning': False,
    },
    'gates': {
        'development_min_macro_f1_gain': DEV_MIN_MACRO_F1_GAIN,
        'development_max_electronic_f1_drop': DEV_MAX_ELECTRONIC_F1_DROP,
        'assessment_min_macro_f1_gain': ASSESSMENT_MIN_MACRO_F1_GAIN,
        'assessment_min_folds_won': ASSESSMENT_MIN_FOLDS_WON,
        'assessment_max_electronic_f1_drop': ASSESSMENT_MAX_ELECTRONIC_F1_DROP,
        'assessment_max_single_fold_gain_share': ASSESSMENT_MAX_SINGLE_FOLD_GAIN_SHARE,
    },
    'single_model_contract': {
        'fold_ensembling': False,
        'seed_ensembling': False,
        'probability_blending': False,
        'model_soup': False,
        'specialist_heads': False,
        'test_time_augmentation': False,
        'multi_crop_inference': False,
        'horizontal_flip_inference': False,
        'multiple_backbone_families': False,
        'manual_prediction_correction': False,
    },
    'official_raw_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'official_test_rows': EXPECTED_TEST_ROWS,
}
LOCKED_CONTRACT_SIGNATURE = sha256_json(LOCKED_CONTRACT)

CONFIG_SNAPSHOT = {
    'team_name': TEAM_NAME,
    'backbone_backend': BACKBONE_BACKEND,
    'generate_control_submission_if_rich_rejected': GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED,
    'zip_include_feature_cache': ZIP_INCLUDE_FEATURE_CACHE,
    'feature_batch_size': FEATURE_BATCH_SIZE,
    'feature_loader_workers': FEATURE_LOADER_WORKERS,
    'feature_shard_size': FEATURE_SHARD_SIZE,
    'artifact_dir': str(ARTIFACT_DIR),
    'prior_cache_search_roots': [str(path) for path in PRIOR_CACHE_SEARCH_ROOTS],
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'historical_references': {'v7': V7_REFERENCE, 'v9a': V9A_REFERENCE},
    'test_access_state_at_configuration_time': dict(TEST_ACCESS_STATE),
}

write_json_atomic({'environment': ENVIRONMENT_INFO, 'gpus': GPU_RUNTIME_INFO}, ARTIFACT_DIR / 'environment.json')
write_json_atomic(CONFIG_SNAPSHOT, ARTIFACT_DIR / 'config.json')
write_json_atomic(
    {'locked_contract': LOCKED_CONTRACT, 'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'locked_contract.json',
)

print(json.dumps(ENVIRONMENT_INFO, indent=2))
for info in GPU_RUNTIME_INFO:
    print('cuda:{device_id}: {name}, {total_vram_gib:.2f} GiB'.format(**info))
print('Manual test-label guard passed: no such CSV attached.')
print('Locked contract signature:', LOCKED_CONTRACT_SIGNATURE)

## 9. Token Hugging Face dan revisi model yang tidak dapat berubah

Checkpoint DINOv3 bersifat *gated*. Token diambil dari `os.environ['HF_TOKEN']` atau dari Kaggle Secret bernama `HF_TOKEN`, tidak pernah dicetak, dan dihapus dari namespace setelah model dimuat.

Alih-alih memakai `main`, notebook me-*resolve* commit SHA 40 karakter yang persis dan memakainya untuk seluruh unduhan. SHA itu dicatat pada `model_manifest.json` dan menjadi bagian dari tanda tangan cache: cache yang dibuat pada revisi lain tidak akan pernah dipakai ulang. Setiap mode kegagalan (token hilang, lisensi belum disetujui, repo tidak dapat diunduh, revisi tidak dapat di-*resolve*) memberi pesan yang langsung dapat ditindaklanjuti.


In [ ]:
from huggingface_hub import model_info, snapshot_download

HF_ACTION_HINT = (
    'Actions to resolve:\n'
    '  1. Create a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) holding a Hugging Face read token.\n'
    '  2. Open https://huggingface.co/{} with that same account and accept the DINOv3 license.\n'
    '  3. Enable notebook internet access.\n'
    '  4. Do not downgrade to a smaller backbone; the locked protocol requires DINOv3 ViT-7B.'
)


def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError(
            'Missing Hugging Face token for the gated DINOv3 checkpoint.\n'
            + HF_ACTION_HINT.format(ACTIVE_MODEL_REPO)
        )
    return token


ACTIVE_MODEL_REPO = TIMM_HF_REPO_ID if BACKBONE_BACKEND == 'timm' else TRANSFORMERS_MODEL_ID
HF_TOKEN_VALUE = resolve_hf_token()
print('Hugging Face authentication present; the token is never printed or persisted.')

try:
    RESOLVED_REVISION = model_info(ACTIVE_MODEL_REPO, revision=MODEL_REVISION_REQUEST, token=HF_TOKEN_VALUE).sha
except Exception as exc:
    raise RuntimeError(
        'Could not resolve an immutable revision for {}. This usually means the license was not accepted, '
        'the token lacks access, or the notebook has no internet.\n'.format(ACTIVE_MODEL_REPO)
        + HF_ACTION_HINT.format(ACTIVE_MODEL_REPO)
    ) from exc
if not RESOLVED_REVISION or len(RESOLVED_REVISION) != 40 or any(
    character not in '0123456789abcdef' for character in RESOLVED_REVISION.lower()
):
    raise RuntimeError(
        'The Hub did not return an immutable 40-character commit SHA for {} (got {!r}).'.format(
            ACTIVE_MODEL_REPO, RESOLVED_REVISION
        )
    )

MODEL_DOWNLOAD_STARTED_AT = time.perf_counter()
try:
    MODEL_SNAPSHOT_DIR = snapshot_download(
        repo_id=ACTIVE_MODEL_REPO,
        revision=RESOLVED_REVISION,
        token=HF_TOKEN_VALUE,
        allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'],
    )
except Exception as exc:
    raise RuntimeError(
        'Failed to download {} at revision {}. Check disk space (the checkpoint is about 27 GB), '
        'internet access, and license acceptance.\n'.format(ACTIVE_MODEL_REPO, RESOLVED_REVISION)
        + HF_ACTION_HINT.format(ACTIVE_MODEL_REPO)
    ) from exc
MODEL_DOWNLOAD_SECONDS = time.perf_counter() - MODEL_DOWNLOAD_STARTED_AT

snapshot_files = sorted(path.name for path in Path(MODEL_SNAPSHOT_DIR).iterdir() if path.is_file())
snapshot_bytes = sum(path.stat().st_size for path in Path(MODEL_SNAPSHOT_DIR).rglob('*') if path.is_file())
print('Repository        :', ACTIVE_MODEL_REPO)
print('Immutable revision:', RESOLVED_REVISION)
print('Snapshot files    :', snapshot_files)
print('Snapshot size     : {:.2f} GiB (downloaded in {:.1f} s)'.format(snapshot_bytes / 2 ** 30, MODEL_DOWNLOAD_SECONDS))

## 10. Resolver raw-train ketat, pemuat citra aman, dan kontrak prapemrosesan

Pencarian direktori train dibatasi kedalamannya dan **memangkas setiap folder bernama `test` tanpa mengenumerasinya**. Kandidat diterima hanya bila anak langsungnya persis `0_Recyclable`, `1_Electronic`, `2_Organic`, tidak ada symlink atau entri tak didukung, dan jumlah per kelas persis 9.999 / 3.961 / 12.567. Bila kandidat yang lolos bukan tepat satu, notebook berhenti.

### Prapemrosesan (sengaja identik dengan V7)

1. Buka citra dengan PIL secara aman, 2. `ImageOps.exif_transpose`, 3. konversi RGB, 4. resize **kotak langsung** ke 256×256, 5. bilinear + antialias, 6. ke tensor, 7. normalisasi ImageNet, 8. hanya tampilan asli, 9. tanpa augmentasi acak, 10. tanpa center crop, 11. tanpa letterbox.

256×256 memang **resolusi native** model ini (`input_size: [3, 256, 256]` pada `config.json` repo timm), jadi tidak ada penskalaan posisi yang aneh. Namun transform default timm memakai **bicubic** dengan `crop_pct = 1.0`, sedangkan V7 memakai **bilinear**. Notebook sengaja **tidak** memakai transform default timm dan mempertahankan pipeline resize V7 persis, supaya perbandingan `timm_control` vs `rich_spatial_fusion` — dan terhadap angka historis V7 — hanya mengubah *representasi*, bukan *geometri*. Mean/std timm tetap diverifikasi cocok dengan konstanta ImageNet yang dipakai; interpolasi bicubic dicatat sebagai **ablasi masa depan yang terdokumentasi**, bukan kandidat pada notebook ini.

Guard memori dekoder: citra dengan header ≥ 8 MP atau sisi > 4096 px diturunkan lebih dulu ke ≤ 1024 px dengan LANCZOS di memori (V7 melakukan hal yang sama lewat salinan JPEG q=95 di disk; versi di memori menghindari 26 ribu penulisan berkas dan satu putaran re-encode JPEG). Dimensi citra hanya dipakai untuk keputusan itu dan tidak pernah menjadi fitur.


In [ ]:
def discover_named_train_dirs(base, max_depth=3):
    if not base.is_dir():
        return []
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            lowered = child.name.lower()
            if lowered == 'test':
                # Pruned without enumeration: official test images stay untouched during selection.
                continue
            if lowered == 'train':
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(train_dir):
    if not train_dir.is_dir():
        return None
    try:
        root_entries = list(train_dir.iterdir())
        if {path.name for path in root_entries} != set(CLASS_DIR_TO_LABEL):
            return None
        if any(path.is_symlink() or not path.is_dir() for path in root_entries):
            return None
        counts = {}
        for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
            entries = list((train_dir / class_dir_name).iterdir())
            if any(
                entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
                for entry in entries
            ):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_RAW_COUNTS else None
    except OSError:
        return None


def resolve_train_dir():
    candidates = [Path.cwd() / 'data' / 'train', KAGGLE_WORKING / 'data' / 'train']
    candidates.extend(discover_named_train_dirs(KAGGLE_INPUT, max_depth=3))
    matches = {
        candidate.resolve()
        for candidate in candidates
        if exact_raw_train_signature(candidate) == EXPECTED_RAW_COUNTS
    }
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official raw train directory with class counts {}; found {}:\n{}\n'
            'Attach the official BDC Satria Data 2026 dataset so that a single "train" folder contains '
            '0_Recyclable, 1_Electronic and 2_Organic with the official counts.'.format(
                EXPECTED_RAW_COUNTS, len(matches), rendered
            )
        )
    return matches.pop()


def load_rgb_image(path):
    try:
        with Image.open(path) as image:
            source_width, source_height = image.size
            if source_width <= 0 or source_height <= 0:
                raise ValueError('invalid image dimensions')
            needs_downscale = (
                source_width * source_height >= LARGE_IMAGE_MP_THRESHOLD
                or max(source_width, source_height) > LARGE_IMAGE_MAX_SOURCE_SIDE
            )
            image.draft('RGB', (SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE))
            rgb_image = ImageOps.exif_transpose(image).convert('RGB')
            if needs_downscale:
                rgb_image.thumbnail((SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE), Image.Resampling.LANCZOS)
            return rgb_image.copy()
    except Exception as exc:
        raise RuntimeError('Image cannot be decoded: {}'.format(path)) from exc


image_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ImageRows(Dataset):
    """Deterministic, order-preserving image reader for frozen feature extraction."""

    def __init__(self, paths):
        self.paths = [str(path) for path in paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, position):
        return image_transform(load_rgb_image(self.paths[position]))


PREPROCESS_CONFIG = {
    'open': 'PIL, EXIF transpose, RGB conversion',
    'large_image_guard': {
        'megapixel_threshold': LARGE_IMAGE_MP_THRESHOLD,
        'max_source_side': LARGE_IMAGE_MAX_SOURCE_SIDE,
        'downscaled_to_max_side': SAFE_DECODE_MAX_SIDE,
        'resample': 'LANCZOS',
        'purpose': 'decoder memory safety only, never a predictive feature',
    },
    'resize_geometry': 'direct_square',
    'spatial_override': {'height': IMAGE_SIZE, 'width': IMAGE_SIZE},
    'interpolation': 'bilinear',
    'antialias': True,
    'rescale_factor': 1.0 / 255.0,
    'image_mean': list(IMAGENET_MEAN),
    'image_std': list(IMAGENET_STD),
    'center_crop': False,
    'letterbox': False,
    'augmentation': 'none',
    'feature_view': FEATURE_VIEW,
    'deviation_from_timm_default': (
        'timm resolves bicubic interpolation with crop_pct 1.0; V10 deliberately keeps the V7 '
        'bilinear direct-square resize so the comparison isolates representation, not geometry'
    ),
    'documented_future_ablation': 'model-native bicubic preprocessing, not implemented as a candidate here',
}
PREPROCESS_SIGNATURE = sha256_json(PREPROCESS_CONFIG)

TRAIN_DIR = resolve_train_dir()
print('Official raw train root:', TRAIN_DIR)
print('Preprocess signature   :', PREPROCESS_SIGNATURE)

## 11. Indeks train, audit jumlah kelas, lima fold stabil, dan tanda tangan data

Baris diurutkan secara deterministik menurut ID citra relatif resmi (`<folder kelas>/<nama berkas>`), label diambil **hanya** dari folder kelas resmi, lalu `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` dijalankan pada urutan itu — persis definisi fold V7, sehingga pembagian fold dapat direproduksi.

Tidak ada baris train yang dibuang atau di-*relabel*. Validasi selalu memakai label asli resmi. Tanda tangan yang dihitung: ID terurut, label, ukuran berkas (identitas raw), dan penugasan fold. Semuanya masuk ke `data_contract.json`, `fold_signature.json`, dan `fold_assignments.csv`, lalu ikut dalam identitas cache.


In [ ]:
def index_raw_train(train_dir):
    rows = []
    observed_counts = {}
    for class_dir_name, label in CLASS_DIR_TO_LABEL.items():
        class_dir = train_dir / class_dir_name
        entries = sorted(class_dir.iterdir(), key=lambda path: path.name)
        invalid = [
            entry.name
            for entry in entries
            if entry.is_symlink() or not entry.is_file() or entry.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES
        ]
        if invalid:
            raise RuntimeError('Nested, unsupported or symlink entries in {}: {}'.format(class_dir, invalid[:10]))
        observed_counts[label] = len(entries)
        for path in entries:
            relative_id = path.relative_to(train_dir).as_posix()
            rows.append({
                'image_id': relative_id,
                'label': label,
                'class_name': CLASS_NAMES[label],
                'path': str(path.resolve()),
                'file_size_bytes': path.stat().st_size,
            })
    if observed_counts != EXPECTED_RAW_COUNTS:
        raise RuntimeError(
            'Raw class counts do not match the official contract: observed={}, expected={}.'.format(
                observed_counts, EXPECTED_RAW_COUNTS
            )
        )
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_TRAIN_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Raw-train row count or image_id uniqueness failed.')
    return frame


train_df = index_raw_train(TRAIN_DIR)
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold_index, (_, valid_positions) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_positions, 'fold'] = fold_index
train_df['fold'] = train_df['fold'].astype(np.int64)
if set(train_df['fold'].tolist()) != set(range(N_FOLDS)) or (train_df['fold'] < 0).any():
    raise RuntimeError('Five-fold assignment is incomplete.')
if train_df['label'].value_counts().sort_index().to_dict() != EXPECTED_RAW_COUNTS:
    raise RuntimeError('Full index no longer matches the official raw counts.')

ORDERED_IMAGE_IDS_SHA256 = sha256_json(train_df['image_id'].tolist())
LABELS_SHA256 = sha256_json(train_df['label'].astype(int).tolist())
FOLD_ASSIGNMENT_SHA256 = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
RAW_DATA_SHA256 = sha256_json(train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records'))
FOLD_CLASS_COUNTS = train_df.groupby(['fold', 'label']).size().unstack(fill_value=0)

FOLD_SIGNATURE = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'fold_definition': {
        'implementation': 'sklearn.model_selection.StratifiedKFold',
        'n_splits': N_FOLDS,
        'shuffle': True,
        'random_state': SEED,
        'ordering': 'relative official image_id ascending before the split',
    },
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256,
    'fold_class_counts': {str(fold): row.to_dict() for fold, row in FOLD_CLASS_COUNTS.iterrows()},
    'validation_labels': 'always the official original labels; no drop and no relabel',
}
DATA_CONTRACT = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'canonical_data': 'official raw labeled train images only',
    'raw_train_root': str(TRAIN_DIR),
    'official_class_codes': {str(index): name for index, name in enumerate(CLASS_NAMES)},
    'class_dir_to_label': CLASS_DIR_TO_LABEL,
    'raw_class_counts': {str(key): value for key, value in EXPECTED_RAW_COUNTS.items()},
    'row_count': int(len(train_df)),
    'ordered_image_ids_sha256': ORDERED_IMAGE_IDS_SHA256,
    'labels_sha256': LABELS_SHA256,
    'raw_data_sha256': RAW_DATA_SHA256,
    'fold_signature': FOLD_SIGNATURE,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'preprocess_config': PREPROCESS_CONFIG,
    'external_labeled_images_used': False,
    'metadata_used_as_feature': False,
    'dropped_or_relabeled_rows': 0,
    'previous_feature_caches_reused': 'only when the complete V10 cache signature matches exactly',
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
DATA_CONTRACT_SIGNATURE = sha256_json(DATA_CONTRACT)

write_json_atomic(FOLD_SIGNATURE, ARTIFACT_DIR / 'fold_signature.json')
write_json_atomic(
    {'data_contract': DATA_CONTRACT, 'data_contract_signature': DATA_CONTRACT_SIGNATURE},
    ARTIFACT_DIR / 'data_contract.json',
)
write_csv_atomic(train_df[['image_id', 'label', 'class_name', 'fold']], ARTIFACT_DIR / 'fold_assignments.csv')

print('Rows:', len(train_df), '| class counts:', train_df['label'].value_counts().sort_index().to_dict())
print('Data contract signature:', DATA_CONTRACT_SIGNATURE)
display(FOLD_CLASS_COUNTS.rename(columns={index: name for index, name in enumerate(CLASS_NAMES)}))

## 12. Konstruksi ViT-7B timm dan *dispatch* ke dua T4

`timm.create_model(..., pretrained=True)` **tidak** menerima argumen `device_map` gaya Transformers, jadi checkpoint 6,72 miliar parameter tidak bisa dimuat begitu saja pada satu T4. Jalur yang dipakai:

1. Bangun arsitektur di bawah `accelerate.init_empty_weights(include_buffers=False)` sehingga parameter berada di `meta` (nol byte RAM) sementara buffer non-persisten seperti tabel periode RoPE tetap terwujud normal.
2. Serahkan snapshot revisi tetap ke `accelerate.load_checkpoint_and_dispatch` dengan `device_map='balanced'`, `max_memory={0: '13GiB', 1: '13GiB'}`, `dtype=torch.float16`, dan `no_split_module_classes` yang diturunkan dari nama kelas blok transformer sebenarnya sehingga **satu blok tidak akan pernah terbelah antar-GPU**.
3. Muat tensor langsung ke GPU tujuan; host RAM tidak pernah menahan checkpoint 26,9 GB tersebut.

FP16 pada 6,72 miliar parameter adalah ± 13,4 GB, jadi setiap T4 memegang ± 6,7 GB dan masih menyisakan ruang lega untuk aktivasi.

**Audit gagal-tertutup setelah dispatch.** Peta perangkat harus memakai `cuda:0` **dan** `cuda:1` dan tidak boleh mengandung `cpu`, `disk`, atau `meta`; tidak boleh ada parameter atau buffer persisten yang tertinggal di `meta`; dtype seluruh parameter floating harus FP16 seragam; jumlah parameter yang dapat dilatih harus nol; dan setiap blok harus berada utuh pada satu perangkat. Tidak ada penurunan diam-diam ke model lebih kecil, ke satu GPU, atau ke offload CPU/disk.

**Jalur cadangan.** `BACKBONE_BACKEND = 'transformers'` memuat `facebook/dinov3-vit7b16-pretrain-lvd1689m` lewat jalur Transformers yang sudah terbukti pada V7. Jalur ini **tidak pernah aktif sendiri**: pengguna harus mengubahnya secara eksplisit. Backend ikut masuk ke tanda tangan cache, jadi cache dari kedua backend tidak akan pernah tertukar.


In [ ]:
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

DISPATCH_FAILURE_HINT = (
    'The DINOv3 ViT-7B backbone could not be distributed entirely across the two Tesla T4 GPUs.\n'
    'Checks to run, in order:\n'
    '  1. Confirm the Kaggle accelerator is "GPU T4 x2" and no other process holds GPU memory.\n'
    '  2. Restart the session so both GPUs start empty, then rerun from the top.\n'
    '  3. If accelerate cannot place the timm module graph, set BACKBONE_BACKEND = "transformers" '
    'in the configuration cell. That fallback is explicit and never activates on its own.\n'
    'CPU offload, disk offload, single-GPU placement and smaller backbones are all forbidden.'
)


def normalize_map_device(value):
    if isinstance(value, int):
        return 'cuda:{}'.format(value)
    text = str(value).lower().replace('cuda ', 'cuda:')
    if text in {'0', '1'}:
        return 'cuda:' + text
    return text


def audit_dispatched_backbone(model, blocks):
    raw_device_map = getattr(model, 'hf_device_map', None)
    if not isinstance(raw_device_map, dict) or not raw_device_map:
        raise RuntimeError('Accelerate did not attach an hf_device_map.\n' + DISPATCH_FAILURE_HINT)
    device_map = {name: normalize_map_device(value) for name, value in raw_device_map.items()}
    mapped_devices = set(device_map.values())
    forbidden = sorted(value for value in mapped_devices if 'cpu' in value or 'disk' in value or 'meta' in value)
    if forbidden:
        raise RuntimeError('CPU/disk/meta placement detected: {}.\n'.format(forbidden) + DISPATCH_FAILURE_HINT)
    if mapped_devices != {'cuda:0', 'cuda:1'}:
        raise RuntimeError(
            'Device map must span exactly cuda:0 and cuda:1; got {}.\n'.format(sorted(mapped_devices))
            + DISPATCH_FAILURE_HINT
        )

    named_tensors = list(model.named_parameters()) + list(model.named_buffers())
    meta_tensors = [name for name, tensor in named_tensors if tensor.device.type == 'meta']
    if meta_tensors:
        raise RuntimeError(
            'These tensors were never materialised and are still on the meta device: {}.\n'.format(meta_tensors[:20])
            + DISPATCH_FAILURE_HINT
        )
    parameter_devices = {str(parameter.device) for parameter in model.parameters()}
    if parameter_devices != {'cuda:0', 'cuda:1'}:
        raise RuntimeError(
            'Parameters are not split across exactly both GPUs: {}.\n'.format(sorted(parameter_devices))
            + DISPATCH_FAILURE_HINT
        )
    parameter_dtypes = {parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()}
    if parameter_dtypes != {BACKBONE_DTYPE}:
        raise RuntimeError('Floating parameters are not uniformly FP16: {}.'.format(sorted(map(str, parameter_dtypes))))
    for module_name, module in model.named_modules():
        for buffer_name, buffer in module._buffers.items():
            if buffer is None or buffer_name in module._non_persistent_buffers_set:
                continue
            if str(buffer.device) not in {'cuda:0', 'cuda:1'}:
                raise RuntimeError(
                    'Persistent buffer {}.{} sits on {}.'.format(module_name, buffer_name, buffer.device)
                )
    for block_index, block in enumerate(blocks):
        block_devices = {str(parameter.device) for parameter in block.parameters()}
        if len(block_devices) != 1:
            raise RuntimeError(
                'Transformer block {} is split across devices {}; blocks must stay whole.\n'.format(
                    block_index, sorted(block_devices)
                )
                + DISPATCH_FAILURE_HINT
            )
    forbidden_wrapper_names = {
        ''.join(('Data', 'Parallel')),
        ''.join(('Distributed', 'Data', 'Parallel')),
        ''.join(('Fully', 'Sharded', 'Data', 'Parallel')),
    }
    if type(model).__name__ in forbidden_wrapper_names:
        raise RuntimeError('Replication or distributed wrapper detected; only model sharding is allowed.')
    return device_map


def build_timm_backbone(snapshot_dir):
    with init_empty_weights(include_buffers=False):
        model = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
    blocks = getattr(model, 'blocks', None)
    if blocks is None or len(blocks) < LAST_N_BLOCKS:
        raise RuntimeError(
            'Could not resolve an ordered transformer block list of at least {} blocks on the timm model.'.format(
                LAST_N_BLOCKS
            )
        )
    block_class_name = type(blocks[0]).__name__
    try:
        model = load_checkpoint_and_dispatch(
            model,
            checkpoint=snapshot_dir,
            device_map=SHARDING_STRATEGY,
            max_memory=SHARDING_MAX_MEMORY,
            no_split_module_classes=[block_class_name],
            dtype=BACKBONE_DTYPE,
        )
    except Exception as exc:
        raise RuntimeError(DISPATCH_FAILURE_HINT) from exc
    model.eval().requires_grad_(False)
    return model, list(model.blocks), block_class_name


def build_transformers_backbone(revision, token):
    from transformers import AutoImageProcessor, AutoModel

    try:
        model = AutoModel.from_pretrained(
            TRANSFORMERS_MODEL_ID,
            revision=revision,
            token=token,
            dtype=BACKBONE_DTYPE,
            low_cpu_mem_usage=True,
            device_map=SHARDING_STRATEGY,
            max_memory=SHARDING_MAX_MEMORY,
            attn_implementation=ATTENTION_IMPLEMENTATION,
        )
    except Exception as exc:
        raise RuntimeError(DISPATCH_FAILURE_HINT) from exc
    model.eval().requires_grad_(False)
    depth = int(model.config.num_hidden_layers)
    blocks = None
    for holder in (model, getattr(model, 'model', None), getattr(model, 'vit', None)):
        if holder is None:
            continue
        for attribute in ('layer', 'layers', 'blocks'):
            candidate = getattr(holder, attribute, None)
            if candidate is None:
                candidate = getattr(getattr(holder, 'encoder', None), attribute, None)
            if isinstance(candidate, (nn.ModuleList, list, tuple)) and len(candidate) == depth:
                blocks = list(candidate)
                break
        if blocks is not None:
            break
    if blocks is None:
        raise RuntimeError('Could not resolve the ordered transformer block list on the Transformers model.')
    processor = AutoImageProcessor.from_pretrained(TRANSFORMERS_MODEL_ID, revision=revision, token=token)
    return model, blocks, processor


MODEL_LOAD_STARTED_AT = time.perf_counter()
TRANSFORMERS_PROCESSOR = None
BACKBONE_BLOCK_CLASS = None
if BACKBONE_BACKEND == 'timm':
    BACKBONE, BACKBONE_BLOCKS, BACKBONE_BLOCK_CLASS = build_timm_backbone(MODEL_SNAPSHOT_DIR)
else:
    BACKBONE, BACKBONE_BLOCKS, TRANSFORMERS_PROCESSOR = build_transformers_backbone(RESOLVED_REVISION, HF_TOKEN_VALUE)
    BACKBONE_BLOCK_CLASS = type(BACKBONE_BLOCKS[0]).__name__
MODEL_LOAD_SECONDS = time.perf_counter() - MODEL_LOAD_STARTED_AT
del HF_TOKEN_VALUE

DEVICE_MAP = audit_dispatched_backbone(BACKBONE, BACKBONE_BLOCKS)
BACKBONE_PARAMETER_COUNT = sum(parameter.numel() for parameter in BACKBONE.parameters())
BACKBONE_TRAINABLE_PARAMETER_COUNT = sum(
    parameter.numel() for parameter in BACKBONE.parameters() if parameter.requires_grad
)
if BACKBONE_TRAINABLE_PARAMETER_COUNT != 0:
    raise RuntimeError('The pretrained backbone is not completely frozen.')
if BACKBONE.training:
    raise RuntimeError('The backbone must stay in evaluation mode.')

NUM_TRANSFORMER_BLOCKS = len(BACKBONE_BLOCKS)
LAST_BLOCK_INDICES = tuple(range(NUM_TRANSFORMER_BLOCKS - LAST_N_BLOCKS, NUM_TRANSFORMER_BLOCKS))

# Token geometry is derived from real model attributes, never assumed.
if BACKBONE_BACKEND == 'timm':
    NUM_PREFIX_TOKENS = getattr(BACKBONE, 'num_prefix_tokens', None)
    if NUM_PREFIX_TOKENS is None:
        raise RuntimeError('The timm model does not expose num_prefix_tokens; the token contract cannot be derived.')
    NUM_PREFIX_TOKENS = int(NUM_PREFIX_TOKENS)
    register_parameter = getattr(BACKBONE, 'reg_token', None)
    NUM_REGISTER_TOKENS = int(register_parameter.shape[1]) if register_parameter is not None else NUM_PREFIX_TOKENS - 1
    HIDDEN_SIZE = int(BACKBONE.num_features)
    FINAL_NORM_MODULE = BACKBONE.norm
    PATCH_INPUT_MODULE = BACKBONE.patch_embed.proj
    TIMM_DATA_CONFIG = timm.data.resolve_model_data_config(BACKBONE)
    resolved_mean = tuple(float(value) for value in TIMM_DATA_CONFIG['mean'])
    resolved_std = tuple(float(value) for value in TIMM_DATA_CONFIG['std'])
    resolved_input_size = tuple(int(value) for value in TIMM_DATA_CONFIG['input_size'])
    if not np.allclose(resolved_mean, IMAGENET_MEAN, atol=1e-6) or not np.allclose(resolved_std, IMAGENET_STD, atol=1e-6):
        raise RuntimeError(
            'timm resolved normalization {} / {} does not match the locked ImageNet constants {} / {}.'.format(
                resolved_mean, resolved_std, IMAGENET_MEAN, IMAGENET_STD
            )
        )
    if resolved_input_size != (3, IMAGE_SIZE, IMAGE_SIZE):
        raise RuntimeError('timm resolved input size {} is not the native {}px contract.'.format(resolved_input_size, IMAGE_SIZE))
    PREPROCESS_RUNTIME_AUDIT = {
        'source': 'timm.data.resolve_model_data_config',
        'resolved_mean': list(resolved_mean),
        'resolved_std': list(resolved_std),
        'resolved_input_size': list(resolved_input_size),
        'resolved_interpolation': TIMM_DATA_CONFIG.get('interpolation'),
        'resolved_crop_pct': TIMM_DATA_CONFIG.get('crop_pct'),
        'applied_interpolation': 'bilinear',
        'timm_default_transform_used': False,
        'reason': 'V7 parity: only the representation changes in this experiment, not the resize geometry',
    }
else:
    backbone_config = BACKBONE.config
    NUM_REGISTER_TOKENS = int(backbone_config.num_register_tokens)
    NUM_PREFIX_TOKENS = 1 + NUM_REGISTER_TOKENS
    HIDDEN_SIZE = int(backbone_config.hidden_size)
    norm_candidates = [
        module
        for module in (
            getattr(BACKBONE, 'norm', None),
            getattr(getattr(BACKBONE, 'model', None), 'norm', None),
        )
        if isinstance(module, nn.Module)
    ]
    unique_norms = {id(module): module for module in norm_candidates}
    if len(unique_norms) != 1:
        raise RuntimeError('Could not uniquely resolve the official final norm on the Transformers model.')
    FINAL_NORM_MODULE = next(iter(unique_norms.values()))
    patch_candidates = [
        module
        for name, module in BACKBONE.named_modules()
        if (name.endswith('patch_embeddings') or name.endswith('patch_embed'))
        and any(True for _ in module.parameters(recurse=True))
    ]
    if len(patch_candidates) != 1:
        raise RuntimeError('Could not uniquely resolve the patch embedding module on the Transformers model.')
    PATCH_INPUT_MODULE = getattr(patch_candidates[0], 'projection', patch_candidates[0])
    processor_mean = tuple(float(value) for value in TRANSFORMERS_PROCESSOR.image_mean)
    processor_std = tuple(float(value) for value in TRANSFORMERS_PROCESSOR.image_std)
    if not np.allclose(processor_mean, IMAGENET_MEAN, atol=1e-6) or not np.allclose(processor_std, IMAGENET_STD, atol=1e-6):
        raise RuntimeError('Official processor normalization does not match the locked ImageNet constants.')
    PREPROCESS_RUNTIME_AUDIT = {
        'source': 'transformers AutoImageProcessor',
        'resolved_mean': list(processor_mean),
        'resolved_std': list(processor_std),
        'resolved_input_size': [3, IMAGE_SIZE, IMAGE_SIZE],
        'applied_interpolation': 'bilinear',
        'timm_default_transform_used': False,
        'reason': 'explicit non-default fallback backend, same V7 preprocessing',
    }

PATCH_INPUT_DEVICE = next(PATCH_INPUT_MODULE.parameters()).device
MODEL_INPUT_DEVICE = PATCH_INPUT_DEVICE
FINAL_NORM_DEVICE = next(FINAL_NORM_MODULE.parameters()).device
PATCH_TOKEN_COUNT = EXPECTED_PATCH_TOKENS
SEQUENCE_TOKEN_COUNT = NUM_PREFIX_TOKENS + PATCH_TOKEN_COUNT

if HIDDEN_SIZE != EXPECTED_HIDDEN_SIZE:
    raise RuntimeError('Runtime hidden size {} does not match the expected {}.'.format(HIDDEN_SIZE, EXPECTED_HIDDEN_SIZE))
if NUM_PREFIX_TOKENS != EXPECTED_PREFIX_TOKENS or NUM_REGISTER_TOKENS != EXPECTED_REGISTER_TOKENS:
    raise RuntimeError(
        'Runtime prefix/register token counts ({}, {}) do not match the expected ({}, {}).'.format(
            NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, EXPECTED_PREFIX_TOKENS, EXPECTED_REGISTER_TOKENS
        )
    )
if SEQUENCE_TOKEN_COUNT != EXPECTED_SEQUENCE_TOKENS:
    raise RuntimeError('Runtime sequence token count {} does not match the expected {}.'.format(SEQUENCE_TOKEN_COUNT, EXPECTED_SEQUENCE_TOKENS))
if str(MODEL_INPUT_DEVICE) not in {'cuda:0', 'cuda:1'}:
    raise RuntimeError('Patch-embedding input device is not CUDA: {}.'.format(MODEL_INPUT_DEVICE))

MODEL_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'backend': BACKBONE_BACKEND,
    'repository': ACTIVE_MODEL_REPO,
    'immutable_revision': RESOLVED_REVISION,
    'snapshot_dir': str(MODEL_SNAPSHOT_DIR),
    'model_download_seconds': MODEL_DOWNLOAD_SECONDS,
    'model_load_seconds': MODEL_LOAD_SECONDS,
    'timm_model_name': TIMM_MODEL_NAME if BACKBONE_BACKEND == 'timm' else None,
    'model_class': type(BACKBONE).__name__,
    'block_class': BACKBONE_BLOCK_CLASS,
    'parameter_count': BACKBONE_PARAMETER_COUNT,
    'trainable_parameter_count': BACKBONE_TRAINABLE_PARAMETER_COUNT,
    'transformer_block_count': NUM_TRANSFORMER_BLOCKS,
    'last_block_indices': list(LAST_BLOCK_INDICES),
    'hidden_size': HIDDEN_SIZE,
    'num_prefix_tokens': NUM_PREFIX_TOKENS,
    'num_register_tokens': NUM_REGISTER_TOKENS,
    'patch_token_count': PATCH_TOKEN_COUNT,
    'sequence_token_count': SEQUENCE_TOKEN_COUNT,
    'token_layout': 'CLS, register tokens, true patch tokens',
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'model_input_device': str(MODEL_INPUT_DEVICE),
    'final_norm_module_class': type(FINAL_NORM_MODULE).__name__,
    'final_norm_device': str(FINAL_NORM_DEVICE),
    'sharding_strategy': SHARDING_STRATEGY,
    'requested_max_memory': {str(key): value for key, value in SHARDING_MAX_MEMORY.items()},
    'cpu_or_disk_offload': False,
    'data_parallel_used': False,
    'preprocess_runtime_audit': PREPROCESS_RUNTIME_AUDIT,
    'implementation_notes': {
        'qkv_bias': (
            'The original DINOv3 weights have all-zero QKV projection biases; timm disables qkv_bias '
            'instead of loading zero-valued parameters. Recorded, not corrected.'
        ),
        'rope_period_dtype': (
            'The original model stores RoPE periods through bfloat16 while timm initialises them in float32. '
            'The timm float32 default is kept because no code-grounded evidence requires changing it.'
        ),
        'backend_is_not_the_hypothesis': (
            'Switching from Transformers to timm is an implementation route for richer patch-aware pooling, '
            'not an expected source of accuracy gain.'
        ),
    },
    'environment': ENVIRONMENT_INFO,
    'gpus': GPU_RUNTIME_INFO,
}
write_json_atomic(MODEL_MANIFEST, ARTIFACT_DIR / 'model_manifest.json')
write_json_atomic(
    {'device_map': DEVICE_MAP, 'devices': sorted(set(DEVICE_MAP.values())), 'gpu_memory': gpu_memory_rows()},
    ARTIFACT_DIR / 'device_map.json',
)

print('Backend            :', BACKBONE_BACKEND, '|', type(BACKBONE).__name__)
print('Parameters         : {:,} (trainable {:,})'.format(BACKBONE_PARAMETER_COUNT, BACKBONE_TRAINABLE_PARAMETER_COUNT))
print('Transformer blocks : {} | cached block indices {}'.format(NUM_TRANSFORMER_BLOCKS, list(LAST_BLOCK_INDICES)))
print('Token contract     : {} = {} prefix ({} register) + {} patch, hidden {}'.format(
    SEQUENCE_TOKEN_COUNT, NUM_PREFIX_TOKENS, NUM_REGISTER_TOKENS, PATCH_TOKEN_COUNT, HIDDEN_SIZE
))
print('Device map devices :', sorted(set(DEVICE_MAP.values())))
print('Model input device :', MODEL_INPUT_DEVICE, '| final norm device:', FINAL_NORM_DEVICE)
for row in gpu_memory_rows():
    print('  cuda:{device_id} allocated {allocated_gib:.2f} GiB, reserved {reserved_gib:.2f} GiB'.format(**row))

## 13. Ekstraksi token empat blok terakhir dengan LayerNorm final **tepat sekali**

Untuk backend timm, dipakai `model.forward_intermediates(..., indices=<4 blok terakhir>, norm=True, return_prefix_tokens=True, output_fmt='NLC', intermediates_only=True)`. Ini API ekstraksi fitur resmi timm dan menerapkan `self.norm` — LayerNorm final DINOv3 yang sudah terlatih — **satu kali** pada setiap blok yang diambil, lalu memisahkan token prefix dari token spasial. Karena `forward_features` juga berakhir dengan `self.norm`, blok terakhir dari `forward_intermediates(norm=True)` harus setara dengan keluaran `forward_features`; kesetaraan itu diuji pada preflight.

`forward_head()` **tidak pernah** dipakai: pooling classifier default timm (`global_pool='avg'`) berbeda dari readout manual yang diinginkan.

Untuk backend cadangan Transformers, dipakai pola V7 yang sudah terbukti: hook pada empat blok terakhir menangkap token mentah, tiga blok pertama dilewatkan LayerNorm final resmi, dan blok terakhir memakai `model_output.last_hidden_state` yang sudah ternormalisasi — sehingga normalisasi tidak pernah diterapkan dua kali.

Kedua jalur mengembalikan pasangan `(prefix_tokens[B, 5, 4096], patch_tokens[B, 256, 4096])` per blok. Prefix token 0 adalah CLS; prefix token 1–4 adalah **register token** dan tidak pernah masuk ke statistik patch mana pun.


In [ ]:
def _validate_block_tokens(prefix_tokens, patch_tokens, block_position, batch_size):
    if tuple(prefix_tokens.shape) != (batch_size, NUM_PREFIX_TOKENS, HIDDEN_SIZE):
        raise RuntimeError(
            'Block {} prefix tokens have shape {}, expected {}.'.format(
                block_position, tuple(prefix_tokens.shape), (batch_size, NUM_PREFIX_TOKENS, HIDDEN_SIZE)
            )
        )
    if tuple(patch_tokens.shape) != (batch_size, PATCH_TOKEN_COUNT, HIDDEN_SIZE):
        raise RuntimeError(
            'Block {} true-patch tokens have shape {}, expected {}.'.format(
                block_position, tuple(patch_tokens.shape), (batch_size, PATCH_TOKEN_COUNT, HIDDEN_SIZE)
            )
        )
    if prefix_tokens.dtype != BACKBONE_DTYPE or patch_tokens.dtype != BACKBONE_DTYPE:
        raise RuntimeError(
            'Block {} normalized tokens are not FP16 ({}, {}).'.format(
                block_position, prefix_tokens.dtype, patch_tokens.dtype
            )
        )


def timm_normalized_last_block_tokens(model_input):
    outputs = BACKBONE.forward_intermediates(
        model_input,
        indices=list(LAST_BLOCK_INDICES),
        return_prefix_tokens=True,
        norm=True,
        output_fmt='NLC',
        intermediates_only=True,
    )
    if not isinstance(outputs, (list, tuple)) or len(outputs) != LAST_N_BLOCKS:
        raise RuntimeError(
            'timm forward_intermediates returned {} intermediates, expected {}.'.format(
                len(outputs) if isinstance(outputs, (list, tuple)) else type(outputs).__name__, LAST_N_BLOCKS
            )
        )
    batch_size = model_input.shape[0]
    blocks = []
    for block_position, item in enumerate(outputs):
        if not isinstance(item, (tuple, list)) or len(item) != 2:
            raise RuntimeError(
                'timm forward_intermediates did not return (spatial, prefix) pairs; '
                'return_prefix_tokens=True is required for register-token exclusion.'
            )
        first, second = item
        # The pair order is resolved from the token counts, so it stays correct regardless of timm ordering.
        if first.shape[1] == NUM_PREFIX_TOKENS and second.shape[1] == PATCH_TOKEN_COUNT:
            prefix_tokens, patch_tokens = first, second
        elif second.shape[1] == NUM_PREFIX_TOKENS and first.shape[1] == PATCH_TOKEN_COUNT:
            patch_tokens, prefix_tokens = first, second
        else:
            raise RuntimeError(
                'Cannot identify prefix and spatial tokens from shapes {} and {}; expected {} prefix and {} patch tokens.'.format(
                    tuple(first.shape), tuple(second.shape), NUM_PREFIX_TOKENS, PATCH_TOKEN_COUNT
                )
            )
        _validate_block_tokens(prefix_tokens, patch_tokens, block_position, batch_size)
        blocks.append((prefix_tokens, patch_tokens))
    return blocks


def _capture_raw_last_blocks(model_input):
    captured = {}
    handles = []

    def make_hook(block_index):
        def hook(_module, _inputs, output):
            tokens = output[0] if isinstance(output, (tuple, list)) else output
            if not torch.is_tensor(tokens):
                tokens = getattr(output, 'last_hidden_state', None)
            if not torch.is_tensor(tokens):
                raise RuntimeError('Transformer block hook did not receive a token tensor.')
            if block_index in captured:
                raise RuntimeError('Cached block hook fired more than once: {}.'.format(block_index))
            captured[block_index] = tokens

        return hook

    try:
        for block_index in LAST_BLOCK_INDICES:
            handles.append(BACKBONE_BLOCKS[block_index].register_forward_hook(make_hook(block_index)))
        model_output = BACKBONE(
            pixel_values=model_input, return_dict=True, output_hidden_states=False, output_attentions=False
        )
        official_final = getattr(model_output, 'last_hidden_state', None)
        if not torch.is_tensor(official_final):
            raise RuntimeError('The Transformers backbone did not return last_hidden_state.')
        if set(captured) != set(LAST_BLOCK_INDICES):
            raise RuntimeError('Not every cached block hook fired exactly once: {}.'.format(sorted(captured)))
        return dict(captured), official_final
    finally:
        for handle in handles:
            handle.remove()
        captured.clear()


def transformers_normalized_last_block_tokens(model_input):
    raw_blocks, official_final = _capture_raw_last_blocks(model_input)
    batch_size = model_input.shape[0]
    blocks = []
    for block_position, block_index in enumerate(LAST_BLOCK_INDICES):
        if block_index == LAST_BLOCK_INDICES[-1]:
            # Already carries the official final norm; normalising again would apply it twice.
            normalized = official_final
        else:
            normalized = FINAL_NORM_MODULE(raw_blocks[block_index].to(FINAL_NORM_DEVICE))
        prefix_tokens = normalized[:, :NUM_PREFIX_TOKENS]
        patch_tokens = normalized[:, NUM_PREFIX_TOKENS:]
        _validate_block_tokens(prefix_tokens, patch_tokens, block_position, batch_size)
        blocks.append((prefix_tokens, patch_tokens))
    return blocks


normalized_last_block_tokens = (
    timm_normalized_last_block_tokens if BACKBONE_BACKEND == 'timm' else transformers_normalized_last_block_tokens
)


def final_norm_equivalence_probe(model_input):
    """Compare the manually normalized final block against the model's own normalized output."""
    with torch.inference_mode():
        blocks = normalized_last_block_tokens(model_input)
        prefix_tokens, patch_tokens = blocks[-1]
        manual_final = torch.cat((prefix_tokens, patch_tokens), dim=1)
        if BACKBONE_BACKEND == 'timm':
            reference_final = BACKBONE.forward_features(model_input)
            reference_source = 'timm forward_features'
        else:
            _, reference_final = _capture_raw_last_blocks(model_input)
            reference_source = 'transformers model_output.last_hidden_state'
        reference_final = reference_final.to(manual_final.device)
        if manual_final.shape != reference_final.shape:
            raise RuntimeError(
                'Manual final-block token shape {} differs from the reference {}.'.format(
                    tuple(manual_final.shape), tuple(reference_final.shape)
                )
            )
        manual_fp32 = manual_final.float()
        reference_fp32 = reference_final.float()
        difference = (manual_fp32 - reference_fp32).abs()
        allclose = bool(
            torch.allclose(manual_fp32, reference_fp32, atol=FINAL_NORM_EQUIVALENCE_ATOL, rtol=FINAL_NORM_EQUIVALENCE_RTOL)
        )
        manual_finite = bool(torch.isfinite(manual_fp32).all().item())
        reference_finite = bool(torch.isfinite(reference_fp32).all().item())
        return {
            'pass': bool(allclose and manual_finite and reference_finite),
            'reference_source': reference_source,
            'shape': list(manual_final.shape),
            'dtype': str(manual_final.dtype),
            'manual_finite': manual_finite,
            'reference_finite': reference_finite,
            'max_abs_difference': float(difference.max().item()),
            'mean_abs_difference': float(difference.mean().item()),
            'atol': FINAL_NORM_EQUIVALENCE_ATOL,
            'rtol': FINAL_NORM_EQUIVALENCE_RTOL,
            'final_norm_applied_times': 1,
        }


print('Token extraction bound to:', normalized_last_block_tokens.__name__)

## 14. Pooling kaya: 17 cabang bernama per citra

Seluruh statistik dihitung dalam FP32 pada token yang **sudah ternormalisasi**, lalu disimpan FP16. Tensor patch mentah `[N, 256, 4096]` **tidak pernah** disimpan — hanya ringkasannya.

**Cabang 0–7, lintas empat blok terakhir** (masing-masing CLS lalu rata-rata 256 patch token asli). Register token dikecualikan dari setiap statistik.

**Cabang 8–16, hanya dari blok terakhir yang ternormalisasi:**

| # | Cabang | Definisi |
|---|---|---|
| 8 | `final_patch_std` | `patches.std(dim=1, unbiased=False)` |
| 9 | `final_patch_max` | maksimum per dimensi fitur atas 256 patch |
| 10–12 | `final_top{10,25,50}_salient_mean` | rata-rata `k` patch paling saliens, `k = 26 / 64 / 128` |
| 13–16 | `final_quadrant_*_mean` | rata-rata 64 patch tiap kuadran 8×8 dari grid 16×16 |

**Definisi saliensi.** Peringkat memakai kosinus antara CLS final dan patch final:

```python
cls_unit = F.normalize(final_cls.float(), dim=-1, eps=1e-6)
patch_unit = F.normalize(final_patches.float(), dim=-1, eps=1e-6)
saliency = torch.einsum('bd,bnd->bn', cls_unit, patch_unit)
```

Norma L2 token **tidak** dipakai untuk memilih patch: setelah LayerNorm, norma token menjadi hampir seragam sehingga peringkatnya nyaris tak berarti. `k` adalah bilangan bulat deterministik (26, 64, 128), bukan hasil pembulatan saat runtime.

**Grid.** `[B, 256, 4096]` di-*reshape* menjadi `[B, 16, 16, 4096]` **hanya setelah** preflight membuktikan jumlah token dan urutan grid baris-mayor secara empiris (Bagian 15).

Biaya cache: `26.527 × 17 × 4096 × 2 byte ≈ 3,69 GB` untuk train dan ≈ 203 MB untuk test — nyaman untuk disk dan RAM Kaggle.


In [ ]:
QUADRANT_SLICES = (
    ('final_quadrant_top_left_mean', slice(0, GRID_SIZE // 2), slice(0, GRID_SIZE // 2)),
    ('final_quadrant_top_right_mean', slice(0, GRID_SIZE // 2), slice(GRID_SIZE // 2, GRID_SIZE)),
    ('final_quadrant_bottom_left_mean', slice(GRID_SIZE // 2, GRID_SIZE), slice(0, GRID_SIZE // 2)),
    ('final_quadrant_bottom_right_mean', slice(GRID_SIZE // 2, GRID_SIZE), slice(GRID_SIZE // 2, GRID_SIZE)),
)


def rich_pool_block_tokens(block_tokens):
    """Turn four normalized (prefix, patch) token pairs into the ordered [B, 17, H] branch tensor."""
    if len(block_tokens) != LAST_N_BLOCKS:
        raise RuntimeError('Rich pooling expects exactly {} normalized blocks.'.format(LAST_N_BLOCKS))
    branches = []
    branch_names = []

    for block_position, (prefix_tokens, patch_tokens) in enumerate(block_tokens):
        # Prefix token 0 is CLS; prefix tokens 1..4 are register tokens and are never pooled.
        cls_fp32 = prefix_tokens[:, 0, :].to(POOLING_COMPUTE_DTYPE)
        patches_fp32 = patch_tokens.to(POOLING_COMPUTE_DTYPE)
        branches.append(cls_fp32)
        branch_names.append('block_m{}_cls'.format(LAST_N_BLOCKS - block_position))
        branches.append(patches_fp32.mean(dim=1))
        branch_names.append('block_m{}_patch_mean'.format(LAST_N_BLOCKS - block_position))

    final_prefix, final_patch_tokens = block_tokens[-1]
    final_cls = final_prefix[:, 0, :].to(POOLING_COMPUTE_DTYPE)
    final_patches = final_patch_tokens.to(POOLING_COMPUTE_DTYPE)

    branches.append(final_patches.std(dim=1, unbiased=False))
    branch_names.append('final_patch_std')
    branches.append(final_patches.max(dim=1).values)
    branch_names.append('final_patch_max')

    cls_unit = F.normalize(final_cls, dim=-1, eps=1e-6)
    patch_unit = F.normalize(final_patches, dim=-1, eps=1e-6)
    saliency = torch.einsum('bd,bnd->bn', cls_unit, patch_unit)
    for fraction, top_k in zip(SALIENT_FRACTIONS, SALIENT_TOP_K):
        top_indices = torch.topk(saliency, top_k, dim=1).indices
        gathered = torch.gather(final_patches, 1, top_indices.unsqueeze(-1).expand(-1, -1, HIDDEN_SIZE))
        branches.append(gathered.mean(dim=1))
        branch_names.append('final_top{}_salient_mean'.format(int(round(fraction * 100))))

    grid = final_patches.reshape(final_patches.shape[0], GRID_SIZE, GRID_SIZE, HIDDEN_SIZE)
    for quadrant_name, row_slice, column_slice in QUADRANT_SLICES:
        branches.append(grid[:, row_slice, column_slice, :].mean(dim=(1, 2)))
        branch_names.append(quadrant_name)

    if tuple(branch_names) != BRANCH_NAMES:
        raise RuntimeError(
            'Rich pooling produced branch order {} but the locked contract is {}.'.format(branch_names, list(BRANCH_NAMES))
        )
    stacked = torch.stack(branches, dim=1)
    if stacked.dtype != POOLING_COMPUTE_DTYPE:
        raise RuntimeError('Rich pooling must be computed in FP32.')
    return stacked


def extract_rich_features(pixel_values):
    """pixel_values: [B, 3, 256, 256] float32 on CPU -> [B, 17, 4096] float16 on CPU."""
    if pixel_values.ndim != 4 or tuple(pixel_values.shape[1:]) != (3, IMAGE_SIZE, IMAGE_SIZE):
        raise RuntimeError('Feature extraction expects [B, 3, {}, {}] input, got {}.'.format(
            IMAGE_SIZE, IMAGE_SIZE, tuple(pixel_values.shape)
        ))
    if pixel_values.dtype != torch.float32:
        raise RuntimeError('Feature extraction expects FP32 pixel values, got {}.'.format(pixel_values.dtype))
    with torch.inference_mode():
        model_input = pixel_values.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE, non_blocking=True)
        block_tokens = normalized_last_block_tokens(model_input)
        pooled = rich_pool_block_tokens(block_tokens)
        if tuple(pooled.shape) != (pixel_values.shape[0], NUM_BRANCHES, HIDDEN_SIZE):
            raise RuntimeError('Pooled feature shape {} is invalid.'.format(tuple(pooled.shape)))
        if not torch.isfinite(pooled).all().item():
            raise RuntimeError('Pooled rich features contain NaN or Inf; extraction is aborted.')
        pooled_max_abs = float(pooled.abs().max().item())
        if not math.isfinite(pooled_max_abs) or pooled_max_abs > 1e6:
            raise RuntimeError('Pooled rich feature range is implausible: {}.'.format(pooled_max_abs))
        return pooled.to(device='cpu', dtype=FEATURE_STORAGE_DTYPE)


print('Rich pooling defines {} branches; salient k = {}'.format(NUM_BRANCHES, SALIENT_TOP_K))

## 15. Preflight ketat sebelum ekstraksi penuh

Ekstraksi penuh memakan waktu berjam-jam, jadi setiap asumsi diuji lebih dulu pada batch kecil. Kegagalan pada tahap mana pun menulis `preflight_report.json` lalu menghentikan notebook.

**Tahap A — kontrak token dan LayerNorm final.** Dua citra sintetis deterministik dilewatkan. Diperiksa: 261 token total, 5 prefix, 4 register, 256 patch asli, hidden 4096, FP16, seluruhnya finit; dan blok terakhir hasil normalisasi manual harus cocok dengan keluaran ternormalisasi milik model sendiri dalam toleransi FP16 (`atol = rtol = 2e-3`, dipilih karena epsilon relatif FP16 ≈ 9,8e-4 sehingga selisih pembulatan wajar lolos sedangkan LayerNorm ganda tidak akan pernah lolos).

**Tahap B — urutan grid patch dibuktikan, bukan diasumsikan.** Dua citra probe dikirim: satu terbagi atas/bawah, satu terbagi kiri/kanan. Setelah `reshape` ke `[1, 16, 16, 4096]`, jarak antar-paruh **baris** harus mendominasi pada probe atas/bawah, dan jarak antar-paruh **kolom** harus mendominasi pada probe kiri/kanan. Bila token ternyata tersusun kolom-mayor, uji ini gagal dan kuadran tidak akan pernah dihitung salah secara diam-diam.

**Tahap C — konsistensi pooling.** Cabang 6 dan 7 dari tensor `[B, 17, 4096]` harus persis sama dengan CLS final dan rata-rata patch final yang dihitung langsung. Selain itu dicatat selisih antara rata-rata 256 patch asli dan rata-rata 260 token non-CLS, sebagai bukti terukur bahwa register token benar-benar dikecualikan.

**Tahap D — throughput nyata.** 24 citra train dipilih deterministik (peringkat SHA-256 atas ID relatif; hanya piksel yang masuk ke model) lalu diukur pada `FEATURE_BATCH_SIZE` yang sedang dikonfigurasi. Laporan menyertakan citra/detik dan estimasi jam untuk 26.527 baris. Kehabisan memori GPU memberi pesan yang menyarankan menurunkan `FEATURE_BATCH_SIZE`.


In [ ]:
GRID_ORDER_MIN_RATIO = 2.0  # dominant-axis separation required to accept row-major patch ordering


def deterministic_pattern(width, height, phase):
    x = np.arange(width, dtype=np.uint32)[None, :]
    y = np.arange(height, dtype=np.uint32)[:, None]
    red = np.broadcast_to((x * 7 + phase * 31) % 256, (height, width))
    green = np.broadcast_to((y * 11 + phase * 17) % 256, (height, width))
    checker = (((x // 9 + y // 13 + phase) % 2) * 127).astype(np.uint32)
    blue = (x * 3 + y * 5 + checker + phase * 23) % 256
    return Image.fromarray(np.stack((red, green, blue), axis=2).astype(np.uint8), mode='RGB')


def half_split_pattern(axis):
    array = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
    if axis == 'rows':
        array[IMAGE_SIZE // 2:, :, :] = 255
    else:
        array[:, IMAGE_SIZE // 2:, :] = 255
    return Image.fromarray(array, mode='RGB')


def deterministic_benchmark_paths(frame, count):
    ranked = sorted(
        ((hashlib.sha256(image_id.encode('utf-8')).hexdigest(), path) for image_id, path in zip(frame['image_id'], frame['path'])),
    )
    if count > len(ranked):
        raise RuntimeError('Benchmark sample is larger than the indexed train set.')
    return [path for _, path in ranked[:count]]


def final_grid_from_images(images):
    batch = torch.stack([image_transform(image) for image in images])
    with torch.inference_mode():
        model_input = batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE, non_blocking=True)
        _, final_patches = normalized_last_block_tokens(model_input)[-1]
        grid = final_patches.to(POOLING_COMPUTE_DTYPE).reshape(len(images), GRID_SIZE, GRID_SIZE, HIDDEN_SIZE)
        half = GRID_SIZE // 2
        row_gap = (grid[:, :half].mean(dim=(1, 2)) - grid[:, half:].mean(dim=(1, 2))).norm(dim=-1)
        column_gap = (grid[:, :, :half].mean(dim=(1, 2)) - grid[:, :, half:].mean(dim=(1, 2))).norm(dim=-1)
        return [float(value) for value in row_gap.cpu()], [float(value) for value in column_gap.cpu()]


def run_preflight():
    stage = 'A_token_contract_and_final_norm'
    report = {
        'schema_version': PREFLIGHT_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'pass': False,
        'failed_stage': None,
        'backend': BACKBONE_BACKEND,
        'model_repository': ACTIVE_MODEL_REPO,
        'model_revision': RESOLVED_REVISION,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
        'preprocess_signature': PREPROCESS_SIGNATURE,
        'device_map_devices': sorted(set(DEVICE_MAP.values())),
        'feature_batch_size': FEATURE_BATCH_SIZE,
        'stages': {},
    }
    try:
        synthetic_specs = ((317, 193, 1), (211, 359, 2))
        synthetic_images = [deterministic_pattern(*spec) for spec in synthetic_specs]
        synthetic_batch = torch.stack([image_transform(image) for image in synthetic_images])
        with torch.inference_mode():
            model_input = synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
            block_tokens = normalized_last_block_tokens(model_input)
            token_rows = []
            for block_position, (prefix_tokens, patch_tokens) in enumerate(block_tokens):
                sequence = torch.cat((prefix_tokens, patch_tokens), dim=1)
                if not torch.isfinite(sequence.float()).all().item():
                    raise RuntimeError('Normalized tokens for block position {} are not finite.'.format(block_position))
                token_rows.append({
                    'block_position': block_position,
                    'transformer_block_index': LAST_BLOCK_INDICES[block_position],
                    'sequence_token_count': int(sequence.shape[1]),
                    'prefix_token_count': int(prefix_tokens.shape[1]),
                    'register_token_count': int(prefix_tokens.shape[1] - 1),
                    'true_patch_token_count': int(patch_tokens.shape[1]),
                    'hidden_size': int(sequence.shape[2]),
                    'dtype': str(sequence.dtype),
                })
        for row in token_rows:
            if (
                row['sequence_token_count'] != EXPECTED_SEQUENCE_TOKENS
                or row['prefix_token_count'] != EXPECTED_PREFIX_TOKENS
                or row['register_token_count'] != EXPECTED_REGISTER_TOKENS
                or row['true_patch_token_count'] != EXPECTED_PATCH_TOKENS
                or row['hidden_size'] != EXPECTED_HIDDEN_SIZE
            ):
                raise RuntimeError('Token contract violated at block position {}: {}.'.format(row['block_position'], row))
        equivalence = final_norm_equivalence_probe(synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE))
        report['stages'][stage] = {
            'pass': bool(equivalence['pass']),
            'images': len(synthetic_images),
            'source_shapes_width_height': [[spec[0], spec[1]] for spec in synthetic_specs],
            'blocks': token_rows,
            'final_norm_equivalence': equivalence,
        }
        if not equivalence['pass']:
            raise RuntimeError(
                'The manually normalized final block does not match the model reference within the FP16 tolerance; '
                'the final norm may be applied zero or two times.'
            )

        stage = 'B_patch_grid_ordering'
        row_probe_row_gap, row_probe_column_gap = final_grid_from_images([half_split_pattern('rows')])
        column_probe_row_gap, column_probe_column_gap = final_grid_from_images([half_split_pattern('columns')])
        row_major = (
            row_probe_row_gap[0] > GRID_ORDER_MIN_RATIO * row_probe_column_gap[0]
            and column_probe_column_gap[0] > GRID_ORDER_MIN_RATIO * column_probe_row_gap[0]
        )
        report['stages'][stage] = {
            'pass': bool(row_major),
            'grid_size': GRID_SIZE,
            'minimum_dominance_ratio': GRID_ORDER_MIN_RATIO,
            'top_bottom_probe': {'row_gap': row_probe_row_gap[0], 'column_gap': row_probe_column_gap[0]},
            'left_right_probe': {'row_gap': column_probe_row_gap[0], 'column_gap': column_probe_column_gap[0]},
            'conclusion': 'row-major [B, 16, 16, H] reshape is valid' if row_major else 'patch ordering is not row-major',
        }
        if not row_major:
            raise RuntimeError(
                'The 256 patch tokens are not laid out row-major, so the 16x16 reshape and the quadrant branches '
                'would be geometrically wrong.'
            )

        stage = 'C_rich_pooling_consistency'
        with torch.inference_mode():
            model_input = synthetic_batch.to(device=MODEL_INPUT_DEVICE, dtype=BACKBONE_DTYPE)
            block_tokens = normalized_last_block_tokens(model_input)
            final_prefix, final_patches = block_tokens[-1]
            direct_cls = final_prefix[:, 0, :].to(POOLING_COMPUTE_DTYPE).cpu()
            direct_patch_mean = final_patches.to(POOLING_COMPUTE_DTYPE).mean(dim=1).cpu()
            all_non_cls = torch.cat((final_prefix[:, 1:, :], final_patches), dim=1).to(POOLING_COMPUTE_DTYPE).mean(dim=1).cpu()
        pooled = extract_rich_features(synthetic_batch)
        cls_difference = float((pooled[:, CONTROL_BRANCH_INDICES[0]].to(POOLING_COMPUTE_DTYPE) - direct_cls).abs().max().item())
        mean_difference = float((pooled[:, CONTROL_BRANCH_INDICES[1]].to(POOLING_COMPUTE_DTYPE) - direct_patch_mean).abs().max().item())
        register_exclusion_delta = float((direct_patch_mean - all_non_cls).abs().max().item())
        consistency_pass = (
            tuple(pooled.shape) == (len(synthetic_images), NUM_BRANCHES, HIDDEN_SIZE)
            and pooled.dtype == FEATURE_STORAGE_DTYPE
            and bool(torch.isfinite(pooled.to(POOLING_COMPUTE_DTYPE)).all().item())
            and cls_difference <= POOLING_SELF_CONSISTENCY_ATOL
            and mean_difference <= POOLING_SELF_CONSISTENCY_ATOL
        )
        report['stages'][stage] = {
            'pass': bool(consistency_pass),
            'pooled_shape': list(pooled.shape),
            'pooled_dtype': str(pooled.dtype),
            'branch_names': list(BRANCH_NAMES),
            'control_branch_indices': list(CONTROL_BRANCH_INDICES),
            'cls_branch_max_abs_difference': cls_difference,
            'patch_mean_branch_max_abs_difference': mean_difference,
            'self_consistency_atol': POOLING_SELF_CONSISTENCY_ATOL,
            'register_exclusion_max_abs_delta': register_exclusion_delta,
            'register_exclusion_note': (
                'non-zero delta proves the 4 register tokens really are excluded from the patch mean'
            ),
        }
        if not consistency_pass:
            raise RuntimeError('Rich pooling branches 6 and 7 do not reproduce the direct final CLS / patch mean.')

        stage = 'D_throughput'
        for device_id in (0, 1):
            torch.cuda.reset_peak_memory_stats(device_id)
        sample_paths = deterministic_benchmark_paths(train_df, BENCHMARK_WARMUP_IMAGES + BENCHMARK_TIMED_IMAGES)
        warmup_loader = DataLoader(
            ImageRows(sample_paths[:BENCHMARK_WARMUP_IMAGES]), batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0
        )
        for batch in warmup_loader:
            extract_rich_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_loader = DataLoader(
            ImageRows(sample_paths[BENCHMARK_WARMUP_IMAGES:]), batch_size=FEATURE_BATCH_SIZE, shuffle=False, num_workers=0
        )
        timed_started = time.perf_counter()
        for batch in timed_loader:
            extract_rich_features(batch)
        for device_id in (0, 1):
            torch.cuda.synchronize(device_id)
        timed_seconds = time.perf_counter() - timed_started
        images_per_second = BENCHMARK_TIMED_IMAGES / timed_seconds
        throughput_pass = images_per_second >= MIN_EXTRACTION_IMAGES_PER_SECOND
        report['stages'][stage] = {
            'pass': bool(throughput_pass),
            'sampling': 'sha256-ranked relative train image IDs; only pixels reach the model',
            'warmup_images': BENCHMARK_WARMUP_IMAGES,
            'timed_images': BENCHMARK_TIMED_IMAGES,
            'timed_seconds': timed_seconds,
            'images_per_second': images_per_second,
            'minimum_images_per_second': MIN_EXTRACTION_IMAGES_PER_SECOND,
            'estimated_train_extraction_hours': EXPECTED_TRAIN_ROWS / images_per_second / 3600.0,
            'estimated_test_extraction_minutes': EXPECTED_TEST_ROWS / images_per_second / 60.0,
            'gpu_memory': gpu_memory_rows(),
            'host_rss_gib': host_rss_gib(),
        }
        if not throughput_pass:
            raise RuntimeError(
                'Measured {:.3f} images/s is below the locked {:.3f} images/s gate.'.format(
                    images_per_second, MIN_EXTRACTION_IMAGES_PER_SECOND
                )
            )

        report['pass'] = True
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        return report
    except torch.cuda.OutOfMemoryError as exc:
        report['failed_stage'] = stage
        report['error_type'] = type(exc).__name__
        report['error'] = str(exc)
        report['action'] = 'Lower FEATURE_BATCH_SIZE (try 2, then 1) in the configuration cell and rerun from the top.'
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError(
            'GPU ran out of memory during preflight stage {} at FEATURE_BATCH_SIZE={}. '
            'Lower FEATURE_BATCH_SIZE to 2 or 1 and rerun.'.format(stage, FEATURE_BATCH_SIZE)
        ) from exc
    except Exception as exc:
        report['failed_stage'] = stage
        report['error_type'] = type(exc).__name__
        report['error'] = str(exc)
        report['action'] = 'Resolve the reported stage and rerun. Full feature extraction has not started.'
        write_json_atomic(report, PREFLIGHT_REPORT_PATH)
        raise RuntimeError('Preflight failed at stage {}; inspect {}.'.format(stage, PREFLIGHT_REPORT_PATH)) from exc


PREFLIGHT_REPORT = run_preflight()
PREFLIGHT_SIGNATURE = sha256_json({
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'model_revision': RESOLVED_REVISION,
    'backend': BACKBONE_BACKEND,
    'last_block_indices': list(LAST_BLOCK_INDICES),
    'token_contract': [EXPECTED_SEQUENCE_TOKENS, EXPECTED_PREFIX_TOKENS, EXPECTED_REGISTER_TOKENS, EXPECTED_PATCH_TOKENS, HIDDEN_SIZE],
    'branch_names': list(BRANCH_NAMES),
})
throughput_stage = PREFLIGHT_REPORT['stages']['D_throughput']
print('Preflight passed. Signature:', PREFLIGHT_SIGNATURE)
print('Final-norm equivalence max abs diff: {:.3e}'.format(
    PREFLIGHT_REPORT['stages']['A_token_contract_and_final_norm']['final_norm_equivalence']['max_abs_difference']
))
print('Patch grid ordering:', PREFLIGHT_REPORT['stages']['B_patch_grid_ordering']['conclusion'])
print('Register exclusion delta: {:.4f}'.format(
    PREFLIGHT_REPORT['stages']['C_rich_pooling_consistency']['register_exclusion_max_abs_delta']
))
print('Throughput: {:.3f} img/s -> train ~{:.2f} h, test ~{:.1f} min'.format(
    throughput_stage['images_per_second'],
    throughput_stage['estimated_train_extraction_hours'],
    throughput_stage['estimated_test_extraction_minutes'],
))

## 16. Cache fitur yang dapat dilanjutkan, atomik, dan terikat tanda tangan

Backbone beku itu mahal, jadi setiap citra hanya boleh melewatinya satu kali.

**Identitas cache** mencakup: versi skema, penanda revisi, backend, repo model + commit SHA 40 karakter, versi timm/PyTorch/CUDA, nama GPU, dtype backbone/pooling/penyimpanan, ukuran citra, tampilan, tanda tangan prapemrosesan, indeks empat blok, kontrak token, perilaku dtype periode RoPE, perilaku QKV bias, urutan 17 nama cabang, jumlah cabang, hidden size, serta tanda tangan ID/label/fold. Perbedaan sekecil apa pun mengubah `cache_signature`, sehingga **cache V7 lama atau cache dengan definisi cabang berbeda tidak mungkin terpakai**.

**Per shard** (256 baris): tensor fitur `[rows, 17, 4096]` FP16, ID citra, label, fold, dan blok metadata lengkap. Shard ditulis atomik; validasi mensyaratkan metadata **sama persis**, bentuk/dtype benar, tidak ada NaN/Inf, dan ID/label/fold sejajar. Shard yang rusak atau basi otomatis dihitung ulang; shard valid dipakai kembali. Manifest hanya ditulis setelah seluruh baris tervalidasi, dan cache lengkap yang valid **tidak pernah dihapus otomatis**.

Tensor patch mentah `[N, 256, 4096]` tidak pernah disimpan.

**Catatan backbone.** Backbone sengaja **tidak** dilepas sebelum pelatihan head, karena backbone yang sama dibutuhkan lagi untuk ekstraksi fitur test setelah keputusan seleksi terkunci; melepasnya berarti mengunduh dan men-*dispatch* ulang 26,9 GB di tengah run. Head hanya butuh beberapa ratus MB pada `cuda:0` sementara bobot backbone memakai ± 6,7 GB per GPU, jadi keduanya muat bersamaan. Cache CUDA tetap dibersihkan pada setiap batas fase.


In [ ]:
CACHE_IDENTITY_BASE = {
    'schema_version': CACHE_SCHEMA_VERSION,
    'revision_marker': REVISION_MARKER,
    'identity_scope': 'frozen_backbone_features_only_no_head_hyperparameters',
    'backend': BACKBONE_BACKEND,
    'model_repository': ACTIVE_MODEL_REPO,
    'model_revision': RESOLVED_REVISION,
    'timm_model_name': TIMM_MODEL_NAME if BACKBONE_BACKEND == 'timm' else None,
    'timm_version': ENVIRONMENT_INFO['timm_version'],
    'torch_version': ENVIRONMENT_INFO['torch_version'],
    'cuda_version': ENVIRONMENT_INFO['cuda_version'],
    'accelerate_version': ENVIRONMENT_INFO['accelerate_version'],
    'gpu_names': [info['name'] for info in GPU_RUNTIME_INFO],
    'image_size': IMAGE_SIZE,
    'feature_view': FEATURE_VIEW,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'backbone_dtype': BACKBONE_DTYPE_NAME,
    'pooling_compute_dtype': POOLING_COMPUTE_DTYPE_NAME,
    'feature_storage_dtype': FEATURE_STORAGE_DTYPE_NAME,
    'last_n_blocks': LAST_N_BLOCKS,
    'last_block_indices': list(LAST_BLOCK_INDICES),
    'per_token_normalization': 'official pretrained DINOv3 final norm applied exactly once',
    'token_contract': {
        'sequence_token_count': SEQUENCE_TOKEN_COUNT,
        'prefix_token_count': NUM_PREFIX_TOKENS,
        'register_token_count': NUM_REGISTER_TOKENS,
        'patch_token_count': PATCH_TOKEN_COUNT,
        'hidden_size': HIDDEN_SIZE,
        'register_tokens_excluded': True,
    },
    'branch_names': list(BRANCH_NAMES),
    'branch_count': NUM_BRANCHES,
    'salient_top_k': list(SALIENT_TOP_K),
    'saliency_definition': 'final CLS to final true-patch cosine similarity',
    'rope_period_dtype_behavior': 'timm float32 initialisation kept (original checkpoint stores bfloat16 periods)',
    'qkv_bias_behavior': 'timm disables qkv_bias because the original projection biases are all zero',
    'raw_patch_tokens_stored': False,
    'preflight_signature': PREFLIGHT_SIGNATURE,
}


def build_cache_identity(split, row_signature):
    identity = dict(CACHE_IDENTITY_BASE)
    identity['split'] = split
    identity.update(row_signature)
    return identity


def discover_prior_cache_dirs(split):
    directories = []
    for root in PRIOR_CACHE_SEARCH_ROOTS:
        if not root.is_dir():
            continue
        for pattern in (
            '*/{}/feature_cache/{}'.format(EXPERIMENT_NAME, split),
            '*/*/{}/feature_cache/{}'.format(EXPERIMENT_NAME, split),
        ):
            for candidate in root.glob(pattern):
                if candidate.is_dir():
                    directories.append(candidate)
    return directories


def shard_name(split, start, end):
    return '{}_{:06d}_{:06d}.pt'.format(split, start, end)


def expected_shard_metadata(split, cache_signature, frame, start, end):
    window = frame.iloc[start:end]
    labels = window['label'].astype(int).tolist() if 'label' in window.columns else None
    folds = window['fold'].astype(int).tolist() if 'fold' in window.columns else None
    return {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'cache_signature': cache_signature,
        'split': split,
        'start': int(start),
        'end': int(end),
        'rows': int(end - start),
        'image_ids_sha256': sha256_json(window['image_id'].tolist()),
        'labels_sha256': sha256_json(labels) if labels is not None else None,
        'folds_sha256': sha256_json(folds) if folds is not None else None,
        'feature_shape': [int(end - start), NUM_BRANCHES, HIDDEN_SIZE],
        'feature_dtype': FEATURE_STORAGE_DTYPE_NAME,
        'branch_names': list(BRANCH_NAMES),
    }


def validate_shard(path, expected_metadata, frame, start, end):
    """Every failure raises RuntimeError so build_feature_cache can safely recompute the shard."""
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
    except Exception as exc:
        raise RuntimeError('Feature shard cannot be loaded: {}'.format(path)) from exc
    if not isinstance(payload, dict) or payload.get('metadata') != expected_metadata:
        raise RuntimeError('Feature shard metadata is stale or mismatched: {}'.format(path))
    features = payload.get('features')
    if not torch.is_tensor(features) or tuple(features.shape) != tuple(expected_metadata['feature_shape']):
        raise RuntimeError('Feature shard tensor shape is invalid: {}'.format(path))
    if features.dtype != FEATURE_STORAGE_DTYPE:
        raise RuntimeError('Feature shard dtype is invalid: {}'.format(path))
    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('Feature shard contains NaN or Inf: {}'.format(path))
    window = frame.iloc[start:end]
    if payload.get('image_ids') != window['image_id'].tolist():
        raise RuntimeError('Feature shard image IDs are misaligned: {}'.format(path))
    if 'label' in window.columns:
        persisted_labels = payload.get('labels')
        expected_labels = torch.tensor(window['label'].to_numpy(), dtype=torch.long)
        if not torch.is_tensor(persisted_labels) or not torch.equal(persisted_labels, expected_labels):
            raise RuntimeError('Feature shard labels are missing or misaligned: {}'.format(path))
    if 'fold' in window.columns:
        persisted_folds = payload.get('folds')
        expected_folds = torch.tensor(window['fold'].to_numpy(), dtype=torch.long)
        if not torch.is_tensor(persisted_folds) or not torch.equal(persisted_folds, expected_folds):
            raise RuntimeError('Feature shard folds are missing or misaligned: {}'.format(path))
    return payload


def extract_shard(split, cache_signature, frame, start, end, destination):
    window = frame.iloc[start:end]
    features = torch.empty((end - start, NUM_BRANCHES, HIDDEN_SIZE), dtype=FEATURE_STORAGE_DTYPE)
    loader = DataLoader(
        ImageRows(window['path'].tolist()),
        batch_size=FEATURE_BATCH_SIZE,
        shuffle=False,
        num_workers=FEATURE_LOADER_WORKERS,
        pin_memory=True,
        persistent_workers=False,
    )
    cursor = 0
    for batch in loader:
        pooled = extract_rich_features(batch)
        features[cursor:cursor + pooled.shape[0]] = pooled
        cursor += pooled.shape[0]
        del pooled
    if cursor != end - start:
        raise RuntimeError('Shard extraction produced {} rows, expected {}.'.format(cursor, end - start))
    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('Freshly extracted shard contains NaN or Inf before saving.')
    metadata = expected_shard_metadata(split, cache_signature, frame, start, end)
    payload = {
        'metadata': metadata,
        'features': features,
        'image_ids': window['image_id'].tolist(),
        'labels': torch.tensor(window['label'].to_numpy(), dtype=torch.long) if 'label' in window.columns else None,
        'folds': torch.tensor(window['fold'].to_numpy(), dtype=torch.long) if 'fold' in window.columns else None,
    }
    atomic_torch_save(payload, destination)
    del payload, features
    return validate_shard(destination, metadata, frame, start, end)


def build_feature_cache(split, frame, cache_dir, manifest_path):
    row_signature = {
        'row_count': int(len(frame)),
        'ordered_image_ids_sha256': sha256_json(frame['image_id'].tolist()),
        'labels_sha256': sha256_json(frame['label'].astype(int).tolist()) if 'label' in frame.columns else None,
        'fold_assignment_sha256': FOLD_ASSIGNMENT_SHA256 if 'fold' in frame.columns else None,
    }
    cache_identity = build_cache_identity(split, row_signature)
    cache_signature = sha256_json(cache_identity)

    if manifest_path.exists():
        existing = read_json(manifest_path)
        if existing.get('cache_signature') != cache_signature or existing.get('complete') is not True:
            raise RuntimeError(
                'An existing {} cache manifest at {} is incomplete or was produced under a different contract. '
                'Move it aside explicitly before rerunning; user artifacts are never deleted automatically.'.format(
                    split, manifest_path
                )
            )

    prior_dirs = discover_prior_cache_dirs(split)
    if prior_dirs:
        print('Prior {} cache directories offered for reuse: {}'.format(split, [str(path) for path in prior_dirs]))

    features = torch.empty((len(frame), NUM_BRANCHES, HIDDEN_SIZE), dtype=FEATURE_STORAGE_DTYPE)
    shard_records = []
    reused_rows = 0
    extracted_rows = 0
    extraction_seconds = 0.0
    started_at = time.perf_counter()
    shard_starts = list(range(0, len(frame), FEATURE_SHARD_SIZE))

    for shard_index, start in enumerate(tqdm(shard_starts, desc='{} feature cache'.format(split))):
        end = min(start + FEATURE_SHARD_SIZE, len(frame))
        expected_metadata = expected_shard_metadata(split, cache_signature, frame, start, end)
        name = shard_name(split, start, end)
        destination = cache_dir / name
        payload = None
        status = None
        for candidate in [destination] + [directory / name for directory in prior_dirs]:
            if not candidate.exists():
                continue
            try:
                payload = validate_shard(candidate, expected_metadata, frame, start, end)
                status = 'reused' if candidate == destination else 'reused_from_prior_run'
                break
            except RuntimeError:
                payload = None
        if payload is None:
            shard_started = time.perf_counter()
            payload = extract_shard(split, cache_signature, frame, start, end, destination)
            extraction_seconds += time.perf_counter() - shard_started
            extracted_rows += end - start
            status = 'written'
        else:
            reused_rows += end - start
        features[start:end].copy_(payload['features'])
        rate = extracted_rows / extraction_seconds if extracted_rows and extraction_seconds > 0 else None
        remaining = len(frame) - end
        shard_records.append({
            'shard_index': shard_index,
            'shard_count': len(shard_starts),
            'file_name': name,
            'start': int(start),
            'end': int(end),
            'rows': int(end - start),
            'status': status,
            'elapsed_seconds': time.perf_counter() - started_at,
            'new_extraction_images_per_second': rate,
            'estimated_remaining_hours': (remaining / rate / 3600.0) if rate else None,
            'host_rss_gib': host_rss_gib(),
        })
        del payload
        if shard_index % 10 == 0 or shard_index == len(shard_starts) - 1:
            print('  shard {}/{} {} | written={} reused={} | rate={} img/s | eta={} h | rss={:.2f} GiB'.format(
                shard_index + 1,
                len(shard_starts),
                status,
                extracted_rows,
                reused_rows,
                '{:.3f}'.format(rate) if rate else 'n/a',
                '{:.2f}'.format(shard_records[-1]['estimated_remaining_hours']) if shard_records[-1]['estimated_remaining_hours'] else 'n/a',
                shard_records[-1]['host_rss_gib'],
            ))

    if extracted_rows + reused_rows != len(frame):
        raise RuntimeError('Feature extraction did not account for every {} row.'.format(split))
    if not torch.isfinite(features.to(POOLING_COMPUTE_DTYPE)).all().item():
        raise RuntimeError('The consolidated {} cache contains NaN or Inf.'.format(split))

    manifest = {
        'schema_version': CACHE_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'complete': True,
        'split': split,
        'cache_signature': cache_signature,
        'cache_identity': cache_identity,
        'shard_size': FEATURE_SHARD_SIZE,
        'shard_count': len(shard_records),
        'rows': int(len(frame)),
        'rows_written_this_session': extracted_rows,
        'rows_reused_this_session': reused_rows,
        'feature_tensor_shape': [int(len(frame)), NUM_BRANCHES, HIDDEN_SIZE],
        'feature_bytes': int(features.numel() * features.element_size()),
        'elapsed_seconds_this_session': time.perf_counter() - started_at,
        'new_extraction_seconds': extraction_seconds,
        'new_extraction_images_per_second': (extracted_rows / extraction_seconds) if extraction_seconds > 0 else None,
        'gpu_memory': gpu_memory_rows(),
        'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
        'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
        'shards': shard_records,
    }
    write_json_atomic(manifest, manifest_path)
    return features, manifest


TRAIN_CACHE_MANIFEST_PATH = ARTIFACT_DIR / 'cache_manifest_train.json'
cached_features, TRAIN_CACHE_MANIFEST = build_feature_cache(
    'train', train_df, TRAIN_CACHE_DIR, TRAIN_CACHE_MANIFEST_PATH
)
TRAIN_CACHE_SIGNATURE = TRAIN_CACHE_MANIFEST['cache_signature']
cached_labels = torch.tensor(train_df['label'].to_numpy(), dtype=torch.long)
cached_folds = torch.tensor(train_df['fold'].to_numpy(), dtype=torch.long)
release_cuda_cache()

print('Train cache ready:', tuple(cached_features.shape), cached_features.dtype)
print('Train cache signature:', TRAIN_CACHE_SIGNATURE)
print('Cache size: {:.2f} GiB | written {} rows, reused {} rows'.format(
    TRAIN_CACHE_MANIFEST['feature_bytes'] / 2 ** 30,
    TRAIN_CACHE_MANIFEST['rows_written_this_session'],
    TRAIN_CACHE_MANIFEST['rows_reused_this_session'],
))

## 17. Dataset cache dan tepat dua head kandidat

Tidak ada *head zoo* di sini. Hanya dua arsitektur, dilatih dengan optimizer, jadwal, bobot kelas, dan jumlah epoch yang **sama persis**, sehingga perbandingannya mengisolasi kualitas representasi.

### `timm_control` — padanan V7 di atas cache baru

`concat(cabang 6, cabang 7)` → 8192 → `LayerNorm → Linear(512) → GELU → Dropout(0.40) → Linear(3)`. Sekitar 4,21 juta parameter.

### `rich_spatial_fusion` — mengonsumsi seluruh 17 cabang

`LayerNorm(4096)` per cabang → satu `Linear(4096, 256)` **bersama** → GELU → Dropout(0.10) → konkatenasi 17×256 = 4352 → `LayerNorm → Linear(512) → GELU → Dropout(0.40) → Linear(3)`. Sekitar 3,43 juta parameter, jadi tidak lebih besar dari kontrol.

Bobot proyeksi dibagi karena seluruh cabang berada di ruang fitur DINO yang sama, sementara LayerNorm per cabang membiarkan tiap cabang punya skalanya sendiri. Tidak ada *branch attention*, *residual tower*, *expert*, transformer, lapisan graf, head spesialis, *cosine classifier*, atau model pohon — tidak satu pun artefak yang diperiksa menunjukkan arsitektur pilihan ini tidak valid.


In [ ]:
HEAD_DEVICE = 'cuda:0'


class CachedBranchRows(Dataset):
    """Serves rows of a cached [N, 17, 4096] FP16 branch tensor without copying it."""

    def __init__(self, features, indices, labels=None):
        self.features = features
        self.indices = [int(index) for index in indices]
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        label = int(self.labels[index]) if self.labels is not None else -1
        return self.features[index], label, index


class TimmControlHead(nn.Module):
    """V7-equivalent readout: final-block CLS concatenated with the final true-patch mean."""

    def __init__(self):
        super().__init__()
        self.branch_indices = list(CONTROL_BRANCH_INDICES)
        input_dim = len(self.branch_indices) * HIDDEN_SIZE
        self.network = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, CONTROL_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(CONTROL_DROPOUT),
            nn.Linear(CONTROL_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, features):
        selected = torch.cat([features[:, index] for index in self.branch_indices], dim=1)
        return self.network(selected)


class RichSpatialFusionHead(nn.Module):
    """One regularized head over all 17 named branches: per-branch LayerNorm, shared projection, fusion."""

    def __init__(self):
        super().__init__()
        self.branch_norms = nn.ModuleList([nn.LayerNorm(HIDDEN_SIZE) for _ in range(NUM_BRANCHES)])
        self.shared_projection = nn.Linear(HIDDEN_SIZE, RICH_PROJECTION_DIM)
        self.projection_dropout = nn.Dropout(RICH_PROJECTION_DROPOUT)
        fused_dim = NUM_BRANCHES * RICH_PROJECTION_DIM
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, RICH_FUSION_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(RICH_FUSION_DROPOUT),
            nn.Linear(RICH_FUSION_HIDDEN_DIM, NUM_CLASSES),
        )

    def forward(self, features):
        normalized = torch.stack(
            [norm(features[:, index]) for index, norm in enumerate(self.branch_norms)], dim=1
        )
        projected = self.projection_dropout(F.gelu(self.shared_projection(normalized)))
        return self.classifier(projected.flatten(1))


def build_head(candidate_name):
    if candidate_name == CONTROL_CANDIDATE:
        return TimmControlHead()
    if candidate_name == RICH_CANDIDATE:
        return RichSpatialFusionHead()
    raise KeyError('Unknown candidate: {}'.format(candidate_name))


CANDIDATE_SPECS = {
    CONTROL_CANDIDATE: {
        'purpose': 'timm equivalent of the adopted V7 readout on the new cache',
        'input_branches': list(CONTROL_BRANCH_NAMES),
        'input_dim': len(CONTROL_BRANCH_INDICES) * HIDDEN_SIZE,
        'architecture': 'LayerNorm -> Linear({}) -> GELU -> Dropout({}) -> Linear({})'.format(
            CONTROL_HIDDEN_DIM, CONTROL_DROPOUT, NUM_CLASSES
        ),
    },
    RICH_CANDIDATE: {
        'purpose': 'consume every rich spatial and distributional branch in one regularized head',
        'input_branches': list(BRANCH_NAMES),
        'input_dim': NUM_BRANCHES * RICH_PROJECTION_DIM,
        'architecture': (
            'per-branch LayerNorm({}) -> shared Linear({}, {}) -> GELU -> Dropout({}) -> concat -> '
            'LayerNorm -> Linear({}) -> GELU -> Dropout({}) -> Linear({})'.format(
                HIDDEN_SIZE, HIDDEN_SIZE, RICH_PROJECTION_DIM, RICH_PROJECTION_DROPOUT,
                RICH_FUSION_HIDDEN_DIM, RICH_FUSION_DROPOUT, NUM_CLASSES,
            )
        ),
    },
}
for candidate_name in CANDIDATE_NAMES:
    probe_head = build_head(candidate_name)
    CANDIDATE_SPECS[candidate_name]['parameter_count'] = sum(
        parameter.numel() for parameter in probe_head.parameters()
    )
    with torch.no_grad():
        probe_logits = probe_head(torch.zeros(2, NUM_BRANCHES, HIDDEN_SIZE))
    if tuple(probe_logits.shape) != (2, NUM_CLASSES):
        raise RuntimeError('Head {} produced logits with shape {}.'.format(candidate_name, tuple(probe_logits.shape)))
    del probe_head, probe_logits

print(json.dumps(CANDIDATE_SPECS, indent=2))

## 18. Loop pelatihan dan evaluasi bersama (deterministik dan dapat diaudit)

Protokol default, identik untuk kedua kandidat: seed 42, bobot kelas seimbang `total / (num_classes * class_count)` dari partisi latih yang sedang dipakai, `CrossEntropyLoss` dengan label smoothing 0.05, AdamW `lr = 8e-4`, `weight_decay = 1e-4`, `CosineAnnealingLR(T_max = 15)`, gradient clipping 1.0, maksimum 15 epoch, early stopping patience 5, batch head 256, batch prediksi 512. Checkpoint terbaik dipilih berdasarkan **Macro F1 validasi**; log loss hanya dicatat sebagai diagnostik sekunder. Tidak ada penyetelan ambang: kelas akhir selalu `argmax`.

Setiap evaluasi menyimpan Macro F1, akurasi, log loss, precision/recall/F1 per kelas, confusion matrix, total error, error Recyclable → Organic, Organic → Recyclable, total error R/O, error Electronic, ringkasan confidence, ringkasan margin dua-teratas, riwayat pelatihan, epoch terbaik, jumlah parameter, dan runtime.

`train_full_head` memakai jalur yang sama tetapi tanpa validasi: ia melatih jumlah epoch tetap yang diturunkan dari bukti *cross-validation*, dengan `T_max` yang sama sehingga lintasan learning rate identik dengan run CV sampai epoch tersebut.


In [ ]:
def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Head probabilities contain NaN or Inf.')
    return probabilities


def predict_logits(model, features, indices):
    index_list = [int(index) for index in indices]
    loader = DataLoader(
        CachedBranchRows(features, index_list),
        batch_size=PREDICT_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    logits_parts = []
    observed_indices = []
    model.eval()
    with torch.inference_mode():
        for batch_features, _batch_labels, batch_indices in loader:
            logits = model(batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True))
            if not torch.isfinite(logits).all().item():
                raise RuntimeError('Head logits contain NaN or Inf.')
            logits_parts.append(logits.float().cpu())
            observed_indices.extend(int(index) for index in batch_indices.tolist())
    if observed_indices != index_list:
        raise RuntimeError('Prediction order changed inside the DataLoader.')
    stacked = torch.cat(logits_parts).numpy().astype(np.float64)
    if stacked.shape != (len(index_list), NUM_CLASSES):
        raise RuntimeError('Head prediction matrix has shape {}.'.format(stacked.shape))
    return stacked


def predict_probabilities(model, features, indices):
    return softmax_numpy(predict_logits(model, features, indices))


def distribution_summary(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values):
        return {'count': 0, 'mean': None, 'std': None, 'q10': None, 'median': None, 'q90': None}
    return {
        'count': int(len(values)),
        'mean': float(values.mean()),
        'std': float(values.std()),
        'q10': float(np.quantile(values, 0.10)),
        'median': float(np.quantile(values, 0.50)),
        'q90': float(np.quantile(values, 0.90)),
    }


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric input probabilities are incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(
        labels,
        predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=list(CLASS_NAMES),
        output_dict=True,
        zero_division=0,
    )
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    confidence = top_two[:, 1]
    margin = top_two[:, 1] - top_two[:, 0]
    correct = predictions == labels
    recyclable_to_organic = int(matrix[0, 2])
    organic_to_recyclable = int(matrix[2, 0])
    return {
        'rows': int(len(labels)),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'per_class': {
            name: {
                'precision': float(report[name]['precision']),
                'recall': float(report[name]['recall']),
                'f1': float(report[name]['f1-score']),
                'support': int(report[name]['support']),
            }
            for name in CLASS_NAMES
        },
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'total_errors': int((~correct).sum()),
        'recyclable_to_organic': recyclable_to_organic,
        'organic_to_recyclable': organic_to_recyclable,
        'ro_errors': recyclable_to_organic + organic_to_recyclable,
        'electronic_errors': int(matrix[1].sum() - matrix[1, 1]),
        'confidence_summary': {
            'all': distribution_summary(confidence),
            'correct': distribution_summary(confidence[correct]),
            'errors': distribution_summary(confidence[~correct]),
        },
        'margin_summary': {
            'all': distribution_summary(margin),
            'correct': distribution_summary(margin[correct]),
            'errors': distribution_summary(margin[~correct]),
        },
    }


def predictions_frame(indices, probabilities, candidate_name, stage, validation_fold):
    indices = np.asarray(indices, dtype=np.int64)
    labels = cached_labels[torch.from_numpy(indices)].numpy()
    predictions = probabilities.argmax(axis=1)
    top_two = np.sort(probabilities, axis=1)[:, -2:]
    return pd.DataFrame({
        'row_index': indices,
        'image_id': train_df.iloc[indices]['image_id'].to_numpy(),
        'label': labels,
        'prediction': predictions,
        'prob_0': probabilities[:, 0],
        'prob_1': probabilities[:, 1],
        'prob_2': probabilities[:, 2],
        'confidence': top_two[:, 1],
        'top_two_margin': top_two[:, 1] - top_two[:, 0],
        'correct': labels == predictions,
        'fold': validation_fold,
        'candidate': candidate_name,
        'stage': stage,
    })


def report_frame(per_class, macro_f1, accuracy):
    rows = [{'label': name, **values} for name, values in per_class.items()]
    rows.append({'label': 'macro avg', 'precision': None, 'recall': None, 'f1': macro_f1, 'support': None})
    rows.append({'label': 'accuracy', 'precision': None, 'recall': None, 'f1': accuracy, 'support': None})
    return pd.DataFrame(rows)


def class_weights_from(label_values):
    class_counts = np.bincount(label_values, minlength=NUM_CLASSES)
    if (class_counts <= 0).any():
        raise RuntimeError('A training partition is missing a class: {}.'.format(class_counts.tolist()))
    weights = class_counts.sum() / (NUM_CLASSES * class_counts)
    return class_counts, torch.tensor(weights, dtype=torch.float32, device=HEAD_DEVICE)


def run_training_epochs(model, train_indices, epochs, criterion, optimizer, scheduler, on_epoch_end):
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        CachedBranchRows(cached_features, train_indices, cached_labels),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
    )
    for epoch in range(1, epochs + 1):
        epoch_started = time.perf_counter()
        model.train()
        loss_sum = 0.0
        seen = 0
        learning_rate = optimizer.param_groups[0]['lr']
        for batch_features, batch_labels, _batch_indices in train_loader:
            batch_features = batch_features.to(device=HEAD_DEVICE, dtype=torch.float32, non_blocking=True)
            batch_labels = batch_labels.to(device=HEAD_DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError('Non-finite loss at epoch {}.'.format(epoch))
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_MAX_NORM)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite head gradient norm at epoch {}.'.format(epoch))
            optimizer.step()
            loss_sum += loss.item() * len(batch_labels)
            seen += len(batch_labels)
        scheduler.step()
        if on_epoch_end(epoch, loss_sum / seen, learning_rate, time.perf_counter() - epoch_started):
            break
    del train_loader


def train_head(candidate_name, validation_fold, stage):
    if candidate_name not in CANDIDATE_NAMES or validation_fold not in range(N_FOLDS):
        raise RuntimeError('Unknown candidate or fold requested.')
    fold_values = cached_folds.numpy()
    train_indices = np.flatnonzero(fold_values != validation_fold)
    valid_indices = np.flatnonzero(fold_values == validation_fold)
    expected_training_folds = set(range(N_FOLDS)) - {validation_fold}
    if set(fold_values[train_indices].tolist()) != expected_training_folds or set(fold_values[valid_indices].tolist()) != {validation_fold}:
        raise RuntimeError('Train/validation fold membership is invalid.')

    train_label_values = cached_labels[torch.from_numpy(train_indices)].numpy()
    valid_labels = cached_labels[torch.from_numpy(valid_indices)].numpy()
    class_counts, class_weights = class_weights_from(train_label_values)

    seed_everything(SEED)
    model = build_head(candidate_name).to(HEAD_DEVICE)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    if parameter_count != sum(p.numel() for p in model.parameters() if p.requires_grad):
        raise RuntimeError('All head parameters must be trainable.')
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    state = {'best_macro_f1': -1.0, 'best_log_loss': float('inf'), 'best_epoch': None, 'best_state': None, 'stale': 0}
    history = []
    training_started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        valid_probabilities = predict_probabilities(model, cached_features, valid_indices)
        valid_predictions = valid_probabilities.argmax(axis=1)
        valid_macro_f1 = float(f1_score(valid_labels, valid_predictions, labels=list(range(NUM_CLASSES)), average='macro'))
        valid_log_loss = float(log_loss(valid_labels, valid_probabilities, labels=list(range(NUM_CLASSES))))
        history.append({
            'stage': stage,
            'candidate': candidate_name,
            'validation_fold': validation_fold,
            'seed': SEED,
            'epoch': epoch,
            'train_loss': train_loss,
            'valid_macro_f1': valid_macro_f1,
            'valid_log_loss': valid_log_loss,
            'learning_rate': learning_rate,
            'epoch_seconds': epoch_seconds,
        })
        print('  {} {} fold={} epoch={:02d} loss={:.6f} f1={:.6f} logloss={:.6f}'.format(
            stage, candidate_name, validation_fold, epoch, train_loss, valid_macro_f1, valid_log_loss
        ))
        improved = valid_macro_f1 > state['best_macro_f1'] + 1e-8 or (
            abs(valid_macro_f1 - state['best_macro_f1']) <= 1e-8 and valid_log_loss < state['best_log_loss']
        )
        if improved:
            state['best_macro_f1'] = valid_macro_f1
            state['best_log_loss'] = valid_log_loss
            state['best_epoch'] = epoch
            state['best_state'] = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            state['stale'] = 0
        else:
            state['stale'] += 1
        return state['stale'] >= EARLY_STOPPING_PATIENCE

    run_training_epochs(model, train_indices, MAX_EPOCHS, criterion, optimizer, scheduler, on_epoch_end)
    if state['best_state'] is None:
        raise RuntimeError('Head training produced no valid checkpoint.')
    training_seconds = time.perf_counter() - training_started

    model.load_state_dict(state['best_state'], strict=True)
    final_probabilities = predict_probabilities(model, cached_features, valid_indices)
    metrics = metric_bundle(valid_labels, final_probabilities)
    metrics.update({
        'schema_version': HEAD_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'stage': stage,
        'candidate': candidate_name,
        'candidate_spec': CANDIDATE_SPECS[candidate_name],
        'parameter_count': parameter_count,
        'best_epoch': state['best_epoch'],
        'training_seconds': training_seconds,
        'validation_fold': validation_fold,
        'training_folds': sorted(expected_training_folds),
        'seed': SEED,
        'train_class_counts': class_counts.tolist(),
        'class_weights': class_weights.detach().cpu().tolist(),
        'cache_signature': TRAIN_CACHE_SIGNATURE,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    })

    prediction_table = predictions_frame(valid_indices, final_probabilities, candidate_name, stage, validation_fold)
    run_stem = '{}_{}_fold{}'.format(stage, candidate_name, validation_fold)
    write_csv_atomic(prediction_table, HEADS_DIR / (run_stem + '_predictions.csv'))
    write_csv_atomic(pd.DataFrame(history), HEADS_DIR / (run_stem + '_history.csv'))
    write_csv_atomic(
        report_frame(metrics['per_class'], metrics['macro_f1'], metrics['accuracy']),
        HEADS_DIR / (run_stem + '_classification_report.csv'),
    )
    write_json_atomic(metrics, HEADS_DIR / (run_stem + '_metrics.json'))
    atomic_torch_save(
        {
            'metadata': {
                'schema_version': HEAD_SCHEMA_VERSION,
                'candidate': candidate_name,
                'stage': stage,
                'validation_fold': validation_fold,
                'seed': SEED,
                'cache_signature': TRAIN_CACHE_SIGNATURE,
                'branch_names': list(BRANCH_NAMES),
            },
            'best_epoch': state['best_epoch'],
            'state_dict': state['best_state'],
        },
        HEADS_DIR / (run_stem + '.pt'),
    )

    del model, criterion, optimizer, scheduler
    state['best_state'] = None
    release_cuda_cache()
    return metrics, prediction_table


def train_full_head(candidate_name, epochs):
    """Retrain one head from scratch on every official raw training row, no early stopping."""
    all_indices = np.arange(len(train_df), dtype=np.int64)
    label_values = cached_labels.numpy()
    class_counts, class_weights = class_weights_from(label_values)

    seed_everything(SEED)
    model = build_head(candidate_name).to(HEAD_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LEARNING_RATE, weight_decay=HEAD_WEIGHT_DECAY)
    # T_max stays at MAX_EPOCHS so the learning-rate trajectory matches the cross-validation runs.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    history = []
    started = time.perf_counter()

    def on_epoch_end(epoch, train_loss, learning_rate, epoch_seconds):
        history.append({
            'stage': 'full_training',
            'candidate': candidate_name,
            'seed': SEED,
            'epoch': epoch,
            'train_loss': train_loss,
            'learning_rate': learning_rate,
            'epoch_seconds': epoch_seconds,
        })
        print('  full_training {} epoch={:02d}/{:02d} loss={:.6f} lr={:.6e}'.format(
            candidate_name, epoch, epochs, train_loss, learning_rate
        ))
        return False

    run_training_epochs(model, all_indices, epochs, criterion, optimizer, scheduler, on_epoch_end)
    metadata = {
        'schema_version': HEAD_SCHEMA_VERSION,
        'revision_marker': REVISION_MARKER,
        'candidate': candidate_name,
        'candidate_spec': CANDIDATE_SPECS[candidate_name],
        'trained_on': 'all {} official raw training rows'.format(len(train_df)),
        'epochs': epochs,
        'epoch_selection': 'median best epoch across the selected recipe cross-validation folds',
        'early_stopping_used': False,
        'seed': SEED,
        'learning_rate': HEAD_LEARNING_RATE,
        'weight_decay': HEAD_WEIGHT_DECAY,
        'batch_size': HEAD_BATCH_SIZE,
        'label_smoothing': LABEL_SMOOTHING,
        'gradient_clip_max_norm': GRAD_CLIP_MAX_NORM,
        'scheduler_t_max': MAX_EPOCHS,
        'train_class_counts': class_counts.tolist(),
        'class_weights': class_weights.detach().cpu().tolist(),
        'parameter_count': sum(parameter.numel() for parameter in model.parameters()),
        'training_seconds': time.perf_counter() - started,
        'cache_signature': TRAIN_CACHE_SIGNATURE,
        'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
        'branch_names': list(BRANCH_NAMES),
        'initialised_from_fold_checkpoint': False,
        'fold_checkpoints_averaged': False,
    }
    state_dict = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    del model, criterion, optimizer, scheduler
    release_cuda_cache()
    return state_dict, pd.DataFrame(history), metadata


print('Training utilities ready. Head device:', HEAD_DEVICE)

## 19. Fase 1 — perbandingan pengembangan pada fold 0

Hanya fold 0 yang dilihat pada fase ini. Kedua kandidat dilatih pada fold 1–4 dan dievaluasi pada fold 0. Fold 1–4 tidak pernah dievaluasi sebagai validasi sampai resep terkunci ditulis.

Referensi historis untuk konteks (bukan target dan bukan klaim): V7 mencatat Macro F1 fold 0 = `0.989918` dengan readout dan protokol head yang sama pada backend Transformers. Bila `timm_control` mendarat jauh dari angka itu, penyebabnya adalah perbedaan backend, bukan hipotesis V10 — dan itu sendiri informasi yang berguna.


In [ ]:
DEVELOPMENT_METRICS = {}
DEVELOPMENT_PREDICTIONS = {}
for candidate_name in CANDIDATE_NAMES:
    print('[development] training {} on fold {}'.format(candidate_name, DEVELOPMENT_FOLD))
    metrics, prediction_table = train_head(candidate_name, DEVELOPMENT_FOLD, 'development')
    DEVELOPMENT_METRICS[candidate_name] = metrics
    DEVELOPMENT_PREDICTIONS[candidate_name] = prediction_table

DEVELOPMENT_SUMMARY_COLUMNS = [
    'candidate', 'parameter_count', 'best_epoch', 'macro_f1', 'accuracy', 'log_loss',
    'total_errors', 'recyclable_to_organic', 'organic_to_recyclable', 'ro_errors',
    'electronic_errors', 'f1_recyclable', 'f1_electronic', 'f1_organic', 'training_seconds',
]
development_table = pd.DataFrame([
    {column: DEVELOPMENT_METRICS[candidate_name][column] for column in DEVELOPMENT_SUMMARY_COLUMNS}
    for candidate_name in CANDIDATE_NAMES
])
write_csv_atomic(development_table, ARTIFACT_DIR / 'development_metrics.csv')
write_json_atomic(
    {
        'development_fold': DEVELOPMENT_FOLD,
        'candidates': list(CANDIDATE_NAMES),
        'metrics': DEVELOPMENT_METRICS,
        'historical_reference_v7_fold0_macro_f1': V7_REFERENCE['fold0_macro_f1'],
        'historical_reference_note': 'V7 used the Transformers backend; this is context, not a V10 claim.',
        'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    },
    ARTIFACT_DIR / 'development_metrics.json',
)
display(development_table)

## 20. Gerbang pengembangan dan resep yang dikunci sebelum penilaian

Kandidat kaya lolos gerbang fold 0 **hanya bila seluruh tujuh syarat terpenuhi**:

1. Gain Macro F1 atas kontrol ≥ `0.0015`.
2. Total error berkurang.
3. Total error Recyclable/Organic berkurang.
4. F1 Electronic turun tidak lebih dari `0.0005`.
5. Prediksi dan probabilitas seluruhnya finit.
6. Gain bukan akibat ketidakcocokan urutan baris atau fold — dibuktikan dengan membandingkan `row_index` dan `label` kedua tabel prediksi baris demi baris.
7. Kandidat kaya tidak melanggar kontrak model tunggal.

Gerbang ini **tidak boleh diturunkan setelah melihat hasil**. Bila kandidat kaya menang tipis (misalnya +0.0003), jawabannya adalah menolak, bukan melonggarkan ambang atau membuat head baru sampai ada yang menang. Fold penilaian tidak dibuka untuk penyetelan tambahan.

Sebelum fold 1–4 dievaluasi, arsitektur terpilih, seluruh hyperparameter, urutan cabang, dan keputusan fold 0 diserialkan ke `locked_recipe_before_assessment.json` beserta `recipe_signature`. Setelah itu resep tidak berubah.

Kontrol tetap dijalankan pada fold 1–4 apa pun hasilnya, karena kontrol adalah pembanding wajib pada fase penilaian **dan** resep cadangan yang butuh bukti lima fold untuk memilih jumlah epoch full-training. Menjalankan kontrol bukan bagian dari seleksi: kontrol sudah ditetapkan sepenuhnya sebelum satu pun hasil terlihat.


In [ ]:
control_development = DEVELOPMENT_METRICS[CONTROL_CANDIDATE]
rich_development = DEVELOPMENT_METRICS[RICH_CANDIDATE]
control_predictions = DEVELOPMENT_PREDICTIONS[CONTROL_CANDIDATE]
rich_predictions = DEVELOPMENT_PREDICTIONS[RICH_CANDIDATE]

alignment_ok = (
    control_predictions['row_index'].tolist() == rich_predictions['row_index'].tolist()
    and control_predictions['label'].tolist() == rich_predictions['label'].tolist()
)
probability_columns = ['prob_0', 'prob_1', 'prob_2']
finite_ok = bool(
    np.isfinite(rich_predictions[probability_columns].to_numpy()).all()
    and np.isfinite(control_predictions[probability_columns].to_numpy()).all()
    and rich_predictions['prediction'].isin(range(NUM_CLASSES)).all()
)
single_model_ok = not any(LOCKED_CONTRACT['single_model_contract'].values())

development_gain = rich_development['macro_f1'] - control_development['macro_f1']
electronic_drop = control_development['f1_electronic'] - rich_development['f1_electronic']
DEVELOPMENT_GATE_CHECKS = {
    'macro_f1_gain_at_least_threshold': bool(development_gain >= DEV_MIN_MACRO_F1_GAIN),
    'total_errors_decrease': bool(rich_development['total_errors'] < control_development['total_errors']),
    'ro_errors_decrease': bool(rich_development['ro_errors'] < control_development['ro_errors']),
    'electronic_f1_stable': bool(electronic_drop <= DEV_MAX_ELECTRONIC_F1_DROP),
    'predictions_and_probabilities_finite': finite_ok,
    'no_row_order_or_fold_mismatch': bool(alignment_ok),
    'single_model_contract_respected': bool(single_model_ok),
}
RICH_PASSED_DEVELOPMENT_GATE = all(DEVELOPMENT_GATE_CHECKS.values())

LOCKED_RECIPE = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'locked_before': 'assessment folds {}'.format(list(ASSESSMENT_FOLDS)),
    'selection_source': 'development fold {} only'.format(DEVELOPMENT_FOLD),
    'control_candidate': CONTROL_CANDIDATE,
    'rich_candidate': RICH_CANDIDATE,
    'candidate_specs': CANDIDATE_SPECS,
    'branch_order': list(BRANCH_NAMES),
    'control_branch_indices': list(CONTROL_BRANCH_INDICES),
    'hyperparameters': LOCKED_CONTRACT['head_protocol'],
    'gates': LOCKED_CONTRACT['gates'],
    'development_fold_decision': {
        'control_macro_f1': control_development['macro_f1'],
        'rich_macro_f1': rich_development['macro_f1'],
        'macro_f1_gain': development_gain,
        'control_total_errors': control_development['total_errors'],
        'rich_total_errors': rich_development['total_errors'],
        'control_ro_errors': control_development['ro_errors'],
        'rich_ro_errors': rich_development['ro_errors'],
        'control_f1_electronic': control_development['f1_electronic'],
        'rich_f1_electronic': rich_development['f1_electronic'],
        'electronic_f1_drop': electronic_drop,
        'checks': DEVELOPMENT_GATE_CHECKS,
        'rich_passed_development_gate': RICH_PASSED_DEVELOPMENT_GATE,
    },
    'candidates_to_assess': (
        [CONTROL_CANDIDATE, RICH_CANDIDATE] if RICH_PASSED_DEVELOPMENT_GATE else [CONTROL_CANDIDATE]
    ),
    'cache_signature': TRAIN_CACHE_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
RECIPE_SIGNATURE = sha256_json(LOCKED_RECIPE)
LOCKED_RECIPE_PAYLOAD = {'locked_recipe': LOCKED_RECIPE, 'recipe_signature': RECIPE_SIGNATURE}
write_json_atomic(LOCKED_RECIPE_PAYLOAD, ARTIFACT_DIR / 'locked_recipe_before_assessment.json')

print('Development fold {} results'.format(DEVELOPMENT_FOLD))
print('  control macro F1 : {:.6f} | errors {} | R/O {} | Electronic F1 {:.6f}'.format(
    control_development['macro_f1'], control_development['total_errors'],
    control_development['ro_errors'], control_development['f1_electronic'],
))
print('  rich    macro F1 : {:.6f} | errors {} | R/O {} | Electronic F1 {:.6f}'.format(
    rich_development['macro_f1'], rich_development['total_errors'],
    rich_development['ro_errors'], rich_development['f1_electronic'],
))
print('  gain over control: {:+.6f} (locked gate {:+.6f})'.format(development_gain, DEV_MIN_MACRO_F1_GAIN))
for check_name, passed in DEVELOPMENT_GATE_CHECKS.items():
    print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print('Rich candidate passed the development gate:', RICH_PASSED_DEVELOPMENT_GATE)
print('Recipe signature locked:', RECIPE_SIGNATURE)

## 21. Fase 2 — fold penilaian 1–4 dan keputusan adopsi *train-only*

Kandidat kaya hanya dijalankan pada fold 1–4 bila ia lolos gerbang fold 0. Kontrol selalu dijalankan karena ia adalah pembanding wajib sekaligus resep cadangan.

Kandidat kaya diadopsi **hanya bila seluruh tujuh syarat terpenuhi**:

1. Gain Macro F1 agregat atas kontrol ≥ `0.0015`.
2. Kandidat kaya mengalahkan kontrol pada minimal 3 dari 4 fold penilaian.
3. Total error agregat berkurang.
4. Total error R/O agregat berkurang.
5. F1 Electronic tetap stabil dalam `0.0005`.
6. Perbaikan tidak terkonsentrasi pada satu fold: bagian gain positif terbesar dari satu fold ≤ 75% dari total gain positif.
7. Tidak ada kebocoran maupun akses test — `test_images_accessed` masih `false`.

Bila gerbang gagal: kandidat kaya ditandai **ditolak**, `timm_control` menjadi resep cadangan *train-only*, keduanya **tidak** di-*blend*, tidak ada arsitektur lain yang disetel, dan notebook melanjutkan ke submission kontrol karena `GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED = True`.

Konteks historis: pada V9A, kandidat terbaik memenangi 2 dari 4 fold dengan gain `+0.00014` dan ditolak. Gerbang di sini lebih ketat dan sudah dikunci sebelum satu pun hasil V10 ada.

`selection_decision.json` ditulis di akhir sel ini — **sebelum** satu pun citra test dibuka.


In [ ]:
ASSESSED_CANDIDATES = LOCKED_RECIPE['candidates_to_assess']
ASSESSMENT_RUNS = []
ASSESSMENT_PREDICTIONS = {candidate_name: [] for candidate_name in ASSESSED_CANDIDATES}
for assessment_fold in ASSESSMENT_FOLDS:
    for candidate_name in ASSESSED_CANDIDATES:
        print('[assessment] training {} on fold {}'.format(candidate_name, assessment_fold))
        metrics, prediction_table = train_head(candidate_name, assessment_fold, 'assessment')
        ASSESSMENT_RUNS.append(metrics)
        ASSESSMENT_PREDICTIONS[candidate_name].append(prediction_table)

assessment_per_fold = pd.DataFrame([
    {
        'candidate': metrics['candidate'],
        'validation_fold': metrics['validation_fold'],
        'best_epoch': metrics['best_epoch'],
        'rows': metrics['rows'],
        'macro_f1': metrics['macro_f1'],
        'accuracy': metrics['accuracy'],
        'log_loss': metrics['log_loss'],
        'total_errors': metrics['total_errors'],
        'recyclable_to_organic': metrics['recyclable_to_organic'],
        'organic_to_recyclable': metrics['organic_to_recyclable'],
        'ro_errors': metrics['ro_errors'],
        'electronic_errors': metrics['electronic_errors'],
        'f1_electronic': metrics['f1_electronic'],
        'training_seconds': metrics['training_seconds'],
    }
    for metrics in ASSESSMENT_RUNS
]).sort_values(['candidate', 'validation_fold']).reset_index(drop=True)
write_csv_atomic(assessment_per_fold, ARTIFACT_DIR / 'assessment_per_fold.csv')

expected_assessment_indices = np.flatnonzero(np.isin(cached_folds.numpy(), ASSESSMENT_FOLDS)).tolist()
ASSESSMENT_AGGREGATE = {}
for candidate_name in ASSESSED_CANDIDATES:
    pooled = pd.concat(ASSESSMENT_PREDICTIONS[candidate_name], ignore_index=True).sort_values('row_index').reset_index(drop=True)
    if pooled['row_index'].tolist() != expected_assessment_indices:
        raise RuntimeError('{} assessment predictions are incomplete or misaligned.'.format(candidate_name))
    ASSESSMENT_AGGREGATE[candidate_name] = metric_bundle(
        pooled['label'].to_numpy(dtype=np.int64), pooled[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float64)
    )
    ASSESSMENT_AGGREGATE[candidate_name]['candidate'] = candidate_name
    ASSESSMENT_AGGREGATE[candidate_name]['folds'] = list(ASSESSMENT_FOLDS)

assessment_aggregate_table = pd.DataFrame([
    {
        'candidate': candidate_name,
        'rows': ASSESSMENT_AGGREGATE[candidate_name]['rows'],
        'macro_f1': ASSESSMENT_AGGREGATE[candidate_name]['macro_f1'],
        'accuracy': ASSESSMENT_AGGREGATE[candidate_name]['accuracy'],
        'log_loss': ASSESSMENT_AGGREGATE[candidate_name]['log_loss'],
        'total_errors': ASSESSMENT_AGGREGATE[candidate_name]['total_errors'],
        'ro_errors': ASSESSMENT_AGGREGATE[candidate_name]['ro_errors'],
        'electronic_errors': ASSESSMENT_AGGREGATE[candidate_name]['electronic_errors'],
        'f1_electronic': ASSESSMENT_AGGREGATE[candidate_name]['f1_electronic'],
    }
    for candidate_name in ASSESSED_CANDIDATES
])
write_csv_atomic(assessment_aggregate_table, ARTIFACT_DIR / 'assessment_aggregate.csv')

ADOPTION_GATE_CHECKS = {}
ASSESSMENT_FOLD_GAINS = {}
if RICH_PASSED_DEVELOPMENT_GATE:
    control_aggregate = ASSESSMENT_AGGREGATE[CONTROL_CANDIDATE]
    rich_aggregate = ASSESSMENT_AGGREGATE[RICH_CANDIDATE]
    fold_macro_f1 = {
        candidate_name: {
            int(row['validation_fold']): float(row['macro_f1'])
            for _, row in assessment_per_fold[assessment_per_fold['candidate'] == candidate_name].iterrows()
        }
        for candidate_name in ASSESSED_CANDIDATES
    }
    ASSESSMENT_FOLD_GAINS = {
        str(fold): fold_macro_f1[RICH_CANDIDATE][fold] - fold_macro_f1[CONTROL_CANDIDATE][fold]
        for fold in ASSESSMENT_FOLDS
    }
    gains = list(ASSESSMENT_FOLD_GAINS.values())
    folds_won = sum(1 for gain in gains if gain > 0)
    positive_gain_total = sum(gain for gain in gains if gain > 0)
    concentration = (max(gains) / positive_gain_total) if positive_gain_total > 0 else 1.0
    aggregate_gain = rich_aggregate['macro_f1'] - control_aggregate['macro_f1']
    aggregate_electronic_drop = control_aggregate['f1_electronic'] - rich_aggregate['f1_electronic']
    ADOPTION_GATE_CHECKS = {
        'aggregate_macro_f1_gain_at_least_threshold': bool(aggregate_gain >= ASSESSMENT_MIN_MACRO_F1_GAIN),
        'wins_at_least_three_of_four_folds': bool(folds_won >= ASSESSMENT_MIN_FOLDS_WON),
        'aggregate_total_errors_decrease': bool(rich_aggregate['total_errors'] < control_aggregate['total_errors']),
        'aggregate_ro_errors_decrease': bool(rich_aggregate['ro_errors'] < control_aggregate['ro_errors']),
        'electronic_f1_stable': bool(aggregate_electronic_drop <= ASSESSMENT_MAX_ELECTRONIC_F1_DROP),
        'improvement_not_concentrated_in_one_fold': bool(concentration <= ASSESSMENT_MAX_SINGLE_FOLD_GAIN_SHARE),
        'no_leakage_or_test_access': bool(
            not TEST_ACCESS_STATE['test_images_accessed'] and not TEST_ACCESS_STATE['manual_test_labels_accessed']
        ),
    }
    RICH_ADOPTED = all(ADOPTION_GATE_CHECKS.values())
    ASSESSMENT_SUMMARY = {
        'aggregate_macro_f1_gain': aggregate_gain,
        'folds_won_against_control': folds_won,
        'per_fold_gain': ASSESSMENT_FOLD_GAINS,
        'largest_single_fold_gain_share': concentration,
        'aggregate_electronic_f1_drop': aggregate_electronic_drop,
    }
else:
    RICH_ADOPTED = False
    ASSESSMENT_SUMMARY = {'reason': 'rich candidate was rejected at the development gate and never reached assessment'}

SELECTED_CANDIDATE = RICH_CANDIDATE if RICH_ADOPTED else CONTROL_CANDIDATE
SELECTION_REASON = (
    'rich_spatial_fusion cleared every locked assessment condition'
    if RICH_ADOPTED
    else 'rich_spatial_fusion did not clear a locked gate; timm_control is the train-only fallback recipe'
)

OOF_METRICS = {}
for candidate_name in ASSESSED_CANDIDATES:
    parts = [DEVELOPMENT_PREDICTIONS[candidate_name]] + ASSESSMENT_PREDICTIONS[candidate_name]
    oof = pd.concat(parts, ignore_index=True).sort_values('row_index').reset_index(drop=True)
    if oof['row_index'].tolist() != list(range(len(train_df))):
        raise RuntimeError('{} out-of-fold predictions do not cover every training row exactly once.'.format(candidate_name))
    probabilities = oof[['prob_0', 'prob_1', 'prob_2']].to_numpy(dtype=np.float64)
    OOF_METRICS[candidate_name] = metric_bundle(oof['label'].to_numpy(dtype=np.int64), probabilities)
    write_csv_atomic(oof, ARTIFACT_DIR / 'oof_predictions_{}.csv'.format(candidate_name))
    write_npy_atomic(probabilities.astype(np.float32), ARTIFACT_DIR / 'oof_probabilities_{}.npy'.format(candidate_name))
    write_json_atomic(OOF_METRICS[candidate_name], ARTIFACT_DIR / 'oof_metrics_{}.json'.format(candidate_name))

SELECTION_DECISION = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'selection_basis': 'train_only_cross_validation',
    'development_fold': DEVELOPMENT_FOLD,
    'assessment_folds': list(ASSESSMENT_FOLDS),
    'assessed_candidates': list(ASSESSED_CANDIDATES),
    'recipe_signature': RECIPE_SIGNATURE,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'cache_signature': TRAIN_CACHE_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'development_gate_checks': DEVELOPMENT_GATE_CHECKS,
    'rich_passed_development_gate': RICH_PASSED_DEVELOPMENT_GATE,
    'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'assessment_summary': ASSESSMENT_SUMMARY,
    'assessment_aggregate': ASSESSMENT_AGGREGATE,
    'out_of_fold_metrics': OOF_METRICS,
    'rich_adopted': RICH_ADOPTED,
    'selected_candidate': SELECTED_CANDIDATE,
    'selection_reason': SELECTION_REASON,
    'blending_used': False,
    'additional_architectures_tuned_after_gate': False,
    'generate_control_submission_if_rich_rejected': GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED,
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
    'historical_reference_v9a': V9A_REFERENCE,
}
SELECTION_DECISION_SIGNATURE = sha256_json(SELECTION_DECISION)
write_json_atomic(
    {'selection_decision': SELECTION_DECISION, 'selection_decision_signature': SELECTION_DECISION_SIGNATURE},
    SELECTION_DECISION_PATH,
)

display(assessment_per_fold)
display(assessment_aggregate_table)
if ADOPTION_GATE_CHECKS:
    for check_name, passed in ADOPTION_GATE_CHECKS.items():
        print('  [{}] {}'.format('PASS' if passed else 'FAIL', check_name))
print('Rich adopted        :', RICH_ADOPTED)
print('Selected candidate  :', SELECTED_CANDIDATE, '-', SELECTION_REASON)
for candidate_name in ASSESSED_CANDIDATES:
    print('  five-fold OOF {:<20s} macro F1 {:.7f} | errors {} | R/O {}'.format(
        candidate_name,
        OOF_METRICS[candidate_name]['macro_f1'],
        OOF_METRICS[candidate_name]['total_errors'],
        OOF_METRICS[candidate_name]['ro_errors'],
    ))
print('Historical V7 five-fold OOF macro F1 (Transformers backend): {:.7f}'.format(
    V7_REFERENCE['five_fold_oof_macro_f1']
))
print('selection_decision.json written before any test image was opened. Signature:', SELECTION_DECISION_SIGNATURE)

## 22. Pemilihan jumlah epoch dan retraining head tunggal pada seluruh data latih

Jumlah epoch full-training diambil dari bukti *cross-validation* resep terpilih: **median epoch terbaik** pada lima fold (fold pengembangan + empat fold penilaian), dibulatkan ke bilangan bulat dan dibatasi pada `[1, 15]`. Median dipilih karena tahan terhadap satu fold yang berhenti terlalu cepat atau terlalu lambat.

Head terpilih lalu **diinisialisasi ulang dari nol** dan dilatih pada seluruh 26.527 baris train resmi dengan arsitektur dan hyperparameter yang persis sama. Tidak ada early stopping berbasis informasi test, tidak ada rata-rata checkpoint fold, tidak ada inisialisasi dari checkpoint fold, dan tidak ada penggabungan head kontrol dengan head kaya. Hasilnya **satu** `final_head.pt`.


In [ ]:
selected_best_epochs = [int(DEVELOPMENT_METRICS[SELECTED_CANDIDATE]['best_epoch'])]
selected_best_epochs.extend(
    int(metrics['best_epoch']) for metrics in ASSESSMENT_RUNS if metrics['candidate'] == SELECTED_CANDIDATE
)
if len(selected_best_epochs) != N_FOLDS:
    raise RuntimeError(
        'Expected {} cross-validation best epochs for {}, found {}.'.format(
            N_FOLDS, SELECTED_CANDIDATE, len(selected_best_epochs)
        )
    )
FULL_TRAIN_EPOCHS = int(min(MAX_EPOCHS, max(1, int(round(float(np.median(selected_best_epochs)))))))
print('Selected candidate     :', SELECTED_CANDIDATE)
print('Cross-validation best epochs:', selected_best_epochs)
print('Full-training epochs (median):', FULL_TRAIN_EPOCHS)

FINAL_HEAD_STATE, FINAL_TRAINING_HISTORY, FINAL_TRAINING_METADATA = train_full_head(SELECTED_CANDIDATE, FULL_TRAIN_EPOCHS)
FINAL_TRAINING_METADATA.update({
    'selected_candidate': SELECTED_CANDIDATE,
    'selection_reason': SELECTION_REASON,
    'cross_validation_best_epochs': selected_best_epochs,
    'recipe_signature': RECIPE_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
})
FINAL_HEAD_PATH = ARTIFACT_DIR / 'final_head.pt'
atomic_torch_save({'metadata': FINAL_TRAINING_METADATA, 'state_dict': FINAL_HEAD_STATE}, FINAL_HEAD_PATH)
write_csv_atomic(FINAL_TRAINING_HISTORY, ARTIFACT_DIR / 'final_training_history.csv')

reloaded_checkpoint = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
if reloaded_checkpoint['metadata'] != FINAL_TRAINING_METADATA:
    raise RuntimeError('The persisted final head failed provenance validation.')
if set(reloaded_checkpoint['state_dict']) != set(FINAL_HEAD_STATE):
    raise RuntimeError('The persisted final head state dict does not match the trained head.')
del reloaded_checkpoint

FINAL_HEAD = build_head(SELECTED_CANDIDATE).to(HEAD_DEVICE)
FINAL_HEAD.load_state_dict(FINAL_HEAD_STATE, strict=True)
FINAL_HEAD.eval()
release_cuda_cache()

display(FINAL_TRAINING_HISTORY)
print('One final head trained on all {} rows for {} epochs and saved to {}'.format(
    len(train_df), FULL_TRAIN_EPOCHS, FINAL_HEAD_PATH
))

## 23. Pembukaan isolasi test dan resolver test/template resmi

Ini adalah **titik pertama** dalam notebook di mana citra test boleh disentuh. Enam prasyarat diperiksa lebih dulu:

1. `selection_decision.json` ada di disk.
2. Tanda tangan keputusan cocok dengan konfigurasi yang sedang berjalan.
3. Cache train lengkap.
4. Evaluasi fold selesai (fold pengembangan + empat fold penilaian untuk resep terpilih).
5. Resep terpilih bersifat *immutable* — `recipe_signature` cocok dengan yang tersimpan.
6. `test_images_accessed` masih `false` selama seluruh proses seleksi.

Baru setelah semua terpenuhi, `TEST_ACCESS_STATE['test_images_accessed']` diubah menjadi `true` dan direktori test resmi serta template submission di-*resolve*.

Template adalah sumber kebenaran untuk urutan baris. Direktori test diindeks sebagai `stem -> path`, lalu path diurutkan **mengikuti kolom `id` template**, bukan mengikuti urutan direktori. Nama berkas dipakai semata-mata untuk memasangkan citra dengan baris template; ia tidak pernah menjadi fitur prediktif.


In [ ]:
persisted_decision = read_json(SELECTION_DECISION_PATH) if SELECTION_DECISION_PATH.exists() else {}
TEST_UNLOCK_CHECKS = {
    'selection_decision_file_exists': bool(persisted_decision),
    'selection_decision_signature_matches': (
        persisted_decision.get('selection_decision_signature') == SELECTION_DECISION_SIGNATURE
    ),
    'train_cache_complete': bool(
        TRAIN_CACHE_MANIFEST.get('complete') is True and TRAIN_CACHE_MANIFEST['rows'] == EXPECTED_TRAIN_ROWS
    ),
    'fold_evaluation_complete': len(selected_best_epochs) == N_FOLDS,
    'selected_recipe_immutable': (
        read_json(ARTIFACT_DIR / 'locked_recipe_before_assessment.json').get('recipe_signature') == RECIPE_SIGNATURE
    ),
    # Read from the persisted decision, not from live state, so rerunning this cell stays idempotent.
    'test_untouched_during_selection': (
        persisted_decision.get('selection_decision', {}).get('test_images_accessed') is False
    ),
    'manual_test_labels_untouched': not TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
failed_unlock_checks = [name for name, passed in TEST_UNLOCK_CHECKS.items() if not passed]
if failed_unlock_checks:
    raise RuntimeError(
        'Test isolation cannot be released. Failed prerequisites: {}. '
        'Rerun the train-only sections until every prerequisite passes.'.format(failed_unlock_checks)
    )

# The configured policy decides whether a control-recipe submission may be produced at all.
if not RICH_ADOPTED and not GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED:
    raise RuntimeError(
        'rich_spatial_fusion was rejected and GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED is False, so the '
        'configured policy forbids producing a timm_control submission. The train-only evidence is already '
        'complete in selection_decision.json and the artifact directory; no test image has been opened. '
        'Set GENERATE_CONTROL_SUBMISSION_IF_RICH_REJECTED = True and rerun this cell to build the control '
        'submission, or stop here and keep the existing V7 submission.'
    )

TEST_ACCESS_STATE['test_images_accessed'] = True
print('Test isolation released. All train-only prerequisites passed:')
for check_name in TEST_UNLOCK_CHECKS:
    print('  [PASS]', check_name)
print('Submission policy: selected recipe = {} (rich adopted: {})'.format(SELECTED_CANDIDATE, RICH_ADOPTED))


def discover_named_test_dirs(base, max_depth=3):
    if not base.is_dir():
        return []
    found = []
    stack = [(base, 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir():
                continue
            if child.name.lower() == 'test':
                found.append(child)
            elif depth < max_depth:
                stack.append((child, depth + 1))
    return found


def valid_test_dir(candidate):
    try:
        entries = list(candidate.iterdir())
    except OSError:
        return False
    if len(entries) != EXPECTED_TEST_ROWS:
        return False
    return all(
        entry.is_file() and not entry.is_symlink() and entry.suffix.lower() in ALLOWED_IMAGE_SUFFIXES
        for entry in entries
    )


def resolve_test_dir():
    candidates = [Path.cwd() / 'data' / 'test', KAGGLE_WORKING / 'data' / 'test']
    candidates.extend(discover_named_test_dirs(KAGGLE_INPUT, max_depth=3))
    matches = {candidate.resolve() for candidate in candidates if candidate.is_dir() and valid_test_dir(candidate)}
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches, key=str)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official test directory holding {} images; found {}:\n{}'.format(
                EXPECTED_TEST_ROWS, len(matches), rendered
            )
        )
    return matches.pop()


def resolve_submission_template():
    matches = {}
    for root in (Path.cwd() / 'data', KAGGLE_WORKING / 'data', KAGGLE_INPUT):
        if not root.is_dir():
            continue
        for candidate in root.rglob('*.csv'):
            try:
                frame = pd.read_csv(candidate)
            except Exception:
                continue
            if list(frame.columns) == SUBMISSION_COLUMNS and len(frame) == EXPECTED_TEST_ROWS:
                matches[str(candidate.resolve())] = frame
    if len(matches) != 1:
        rendered = '\n'.join('  - {}'.format(path) for path in sorted(matches)) or '  (none)'
        raise RuntimeError(
            'Expected exactly one official submission template with columns {} and {} rows; found {}:\n{}'.format(
                SUBMISSION_COLUMNS, EXPECTED_TEST_ROWS, len(matches), rendered
            )
        )
    path = next(iter(matches))
    return Path(path), matches[path]


TEST_DIR = resolve_test_dir()
TEMPLATE_PATH, submission_template = resolve_submission_template()

stem_to_path = {}
for path in sorted(TEST_DIR.iterdir(), key=lambda item: item.name):
    if path.stem in stem_to_path:
        raise RuntimeError('Duplicate test image stem: {}'.format(path.stem))
    stem_to_path[path.stem] = path

template_ids = submission_template['id'].tolist()
if len(set(template_ids)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('The official template contains duplicate ids.')
ordered_test_paths = []
for raw_id in template_ids:
    key = Path(str(raw_id)).stem
    if key not in stem_to_path:
        raise RuntimeError('Template id {!r} has no matching image file in {}.'.format(raw_id, TEST_DIR))
    ordered_test_paths.append(str(stem_to_path[key].resolve()))
if len(set(ordered_test_paths)) != EXPECTED_TEST_ROWS:
    raise RuntimeError('Template ids did not map to {} distinct test images.'.format(EXPECTED_TEST_ROWS))

test_df = pd.DataFrame({
    'image_id': [str(raw_id) for raw_id in template_ids],
    'path': ordered_test_paths,
})
TEST_ORDER_SHA256 = sha256_json(test_df['image_id'].tolist())

print('Official test directory  :', TEST_DIR)
print('Official template        :', TEMPLATE_PATH)
print('Test rows in template    :', len(test_df))
print('Template row-order sha256:', TEST_ORDER_SHA256)

## 24. Ekstraksi fitur test, inferensi model tunggal, dan pembuatan submission

Fitur test diekstraksi lewat **fungsi yang sama persis** dengan train: revisi model timm yang sama, prapemrosesan yang sama, ekstraksi token yang sama, definisi cabang yang sama, dtype yang sama, dan skema cache yang sama. Perbedaan satu-satunya adalah `split = 'test'` dan tanda tangan barisnya, yang keduanya masuk ke `cache_manifest_test.json`.

Inferensi adalah **satu forward pass deterministik** dengan **satu** head hasil full-training, dalam mode evaluasi. Matriks probabilitas `[1458, 3]` disimpan sebagai `test_probabilities.npy`, lalu kelas ditentukan dengan `argmax` dan dipetakan ke kode numerik 0/1/2. Tanpa TTA, tanpa multi-crop, tanpa flip, tanpa rata-rata antar-fold, tanpa blending.

Validasi submission yang ditegakkan: tepat 1.458 baris, kolom persis `id,predicted`, tidak ada nilai kosong, tidak ada ID duplikat, seluruh prediksi berada di `{0, 1, 2}`, dan kolom `id` **identik urutannya** dengan template resmi. Validasi format bersifat otomatis; berkas submission tidak pernah dibaca ulang untuk mengubah prediksi secara manual.


In [ ]:
TEST_CACHE_MANIFEST_PATH = ARTIFACT_DIR / 'cache_manifest_test.json'
test_features, TEST_CACHE_MANIFEST = build_feature_cache('test', test_df, TEST_CACHE_DIR, TEST_CACHE_MANIFEST_PATH)
TEST_CACHE_SIGNATURE = TEST_CACHE_MANIFEST['cache_signature']
if tuple(test_features.shape) != (EXPECTED_TEST_ROWS, NUM_BRANCHES, HIDDEN_SIZE):
    raise RuntimeError('Test feature tensor has shape {}.'.format(tuple(test_features.shape)))
train_identity = dict(TRAIN_CACHE_MANIFEST['cache_identity'])
test_identity = dict(TEST_CACHE_MANIFEST['cache_identity'])
shared_keys = set(CACHE_IDENTITY_BASE)
if {key: train_identity[key] for key in shared_keys} != {key: test_identity[key] for key in shared_keys}:
    raise RuntimeError('The train and test caches were not produced under the same frozen-backbone contract.')
release_cuda_cache()

TEST_INFERENCE_STARTED_AT = time.perf_counter()
test_probabilities = predict_probabilities(FINAL_HEAD, test_features, np.arange(len(test_df), dtype=np.int64))
TEST_INFERENCE_SECONDS = time.perf_counter() - TEST_INFERENCE_STARTED_AT
if test_probabilities.shape != (EXPECTED_TEST_ROWS, NUM_CLASSES) or not np.isfinite(test_probabilities).all():
    raise RuntimeError('Test probability matrix is incomplete or non-finite.')
write_npy_atomic(test_probabilities.astype(np.float32), ARTIFACT_DIR / 'test_probabilities.npy')

test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)
submission = pd.DataFrame({'id': submission_template['id'].to_numpy(), 'predicted': test_predictions})

SUBMISSION_VALIDATION = {
    'row_count_is_expected': bool(len(submission) == EXPECTED_TEST_ROWS),
    'columns_are_exact': bool(list(submission.columns) == SUBMISSION_COLUMNS),
    'no_missing_values': bool(not submission.isna().any().any()),
    'ids_are_unique': bool(submission['id'].is_unique),
    'predictions_in_class_codes': bool(submission['predicted'].isin([0, 1, 2]).all()),
    'row_order_matches_template': bool(submission['id'].tolist() == submission_template['id'].tolist()),
}
failed_validation = [name for name, passed in SUBMISSION_VALIDATION.items() if not passed]
if failed_validation:
    raise RuntimeError('Submission validation failed: {}.'.format(failed_validation))

SUBMISSION_PATH = ARTIFACT_DIR / 'submission_{}.csv'.format(sanitize_for_filename(TEAM_NAME))
write_csv_atomic(submission, SUBMISSION_PATH)

TEST_INFERENCE_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'selected_candidate': SELECTED_CANDIDATE,
    'selection_reason': SELECTION_REASON,
    'recipe_signature': RECIPE_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'final_head_checkpoint': str(FINAL_HEAD_PATH),
    'full_training_epochs': FULL_TRAIN_EPOCHS,
    'model_repository': ACTIVE_MODEL_REPO,
    'model_revision': RESOLVED_REVISION,
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'test_cache_signature': TEST_CACHE_SIGNATURE,
    'template_path': str(TEMPLATE_PATH),
    'test_dir': str(TEST_DIR),
    'test_order_sha256': TEST_ORDER_SHA256,
    'inference_passes': 1,
    'test_time_augmentation': False,
    'multi_crop_inference': False,
    'horizontal_flip_inference': False,
    'fold_or_seed_ensembling': False,
    'probability_blending': False,
    'manual_prediction_correction': False,
    'decision_rule': 'argmax over the single probability matrix',
    'inference_seconds': TEST_INFERENCE_SECONDS,
    'submission_path': str(SUBMISSION_PATH),
    'submission_validation': SUBMISSION_VALIDATION,
    'predicted_class_distribution': {
        str(class_index): int((test_predictions == class_index).sum()) for class_index in range(NUM_CLASSES)
    },
    'mean_max_probability': float(test_probabilities.max(axis=1).mean()),
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
    'previous_submissions_loaded': False,
    'leaderboard_feedback_used': False,
}
write_json_atomic(TEST_INFERENCE_MANIFEST, ARTIFACT_DIR / 'test_inference_manifest.json')

for check_name in SUBMISSION_VALIDATION:
    print('  [PASS]', check_name)
print('Predicted class distribution:', TEST_INFERENCE_MANIFEST['predicted_class_distribution'])
print('Mean top-class probability  : {:.5f}'.format(TEST_INFERENCE_MANIFEST['mean_max_probability']))
print('Submission written to       :', SUBMISSION_PATH)
display(submission.head())
if TEAM_NAME == 'CHANGE_ME':
    print('\n' + '!' * 78)
    print('!! TEAM_NAME was never set. Rename the file to submission_<NamaTim>.csv before  !!')
    print('!! uploading, or set TEAM_NAME in the configuration cell and rerun this cell.   !!')
    print('!' * 78)

## 25. Metrik, plot, dan analisis error

Seluruh diagnostik di bawah ini hanya memakai prediksi **out-of-fold pada data latih**. Tidak ada citra test yang dibuka, ditampilkan, atau dianalisis untuk keputusan model apa pun — inferensi test sudah selesai dan tidak akan diubah.

Yang dihasilkan: confusion matrix mentah dan ternormalisasi baris untuk setiap kandidat yang dinilai, kurva Macro F1 validasi per epoch pada fold pengembangan, distribusi confidence dan margin dua-teratas untuk resep terpilih, serta tabel error paling percaya-diri pada batas Recyclable ↔ Organic — yaitu tempat sisa error proyek terkonsentrasi.


In [ ]:
def normalized_confusion(matrix):
    matrix = np.asarray(matrix, dtype=np.float64)
    row_sums = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, row_sums, out=np.zeros_like(matrix), where=row_sums > 0)


figure, axes = plt.subplots(2, len(ASSESSED_CANDIDATES), figsize=(5.5 * len(ASSESSED_CANDIDATES), 9), squeeze=False)
for column, candidate_name in enumerate(ASSESSED_CANDIDATES):
    raw_matrix = np.asarray(OOF_METRICS[candidate_name]['confusion_matrix'])
    for row, (matrix, title, value_format) in enumerate((
        (raw_matrix, 'raw', 'd'),
        (normalized_confusion(raw_matrix), 'row-normalized', '.4f'),
    )):
        axis = axes[row][column]
        image = axis.imshow(matrix, cmap='Blues', vmin=0, vmax=None if row == 0 else 1)
        axis.set_xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=25, ha='right')
        axis.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
        axis.set_xlabel('Predicted')
        axis.set_ylabel('True')
        axis.set_title('{} five-fold OOF ({})'.format(candidate_name, title), fontsize=10)
        for true_label in range(NUM_CLASSES):
            for predicted_label in range(NUM_CLASSES):
                value = matrix[true_label, predicted_label]
                rendered = format(int(value), value_format) if value_format == 'd' else format(value, value_format)
                axis.text(predicted_label, true_label, rendered, ha='center', va='center', fontsize=9)
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'oof_confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(13, 4))
for candidate_name in CANDIDATE_NAMES:
    history_path = HEADS_DIR / 'development_{}_fold{}_history.csv'.format(candidate_name, DEVELOPMENT_FOLD)
    if not history_path.exists():
        continue
    history = pd.read_csv(history_path)
    axes[0].plot(history['epoch'], history['valid_macro_f1'], marker='o', label=candidate_name)
    axes[1].plot(history['epoch'], history['valid_log_loss'], marker='o', label=candidate_name)
axes[0].set(title='Development fold {} validation Macro F1'.format(DEVELOPMENT_FOLD), xlabel='epoch', ylabel='Macro F1')
axes[1].set(title='Development fold {} validation log loss'.format(DEVELOPMENT_FOLD), xlabel='epoch', ylabel='log loss')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'development_training_history.png', dpi=160, bbox_inches='tight')
plt.show()

selected_oof = pd.read_csv(ARTIFACT_DIR / 'oof_predictions_{}.csv'.format(SELECTED_CANDIDATE))
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(selected_oof.loc[selected_oof['correct'], 'confidence'], bins=40, alpha=0.55, density=True, label='correct')
axes[0].hist(selected_oof.loc[~selected_oof['correct'], 'confidence'], bins=40, alpha=0.55, density=True, label='errors')
axes[1].hist(selected_oof.loc[selected_oof['correct'], 'top_two_margin'], bins=40, alpha=0.55, density=True, label='correct')
axes[1].hist(selected_oof.loc[~selected_oof['correct'], 'top_two_margin'], bins=40, alpha=0.55, density=True, label='errors')
axes[0].set(title='{} OOF confidence'.format(SELECTED_CANDIDATE), xlabel='maximum probability', ylabel='density')
axes[1].set(title='{} OOF top-two margin'.format(SELECTED_CANDIDATE), xlabel='margin', ylabel='density')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'oof_confidence_and_margin.png', dpi=160, bbox_inches='tight')
plt.show()

selected_errors = selected_oof.loc[~selected_oof['correct']].copy()
recyclable_organic_mask = (
    ((selected_errors['label'] == 0) & (selected_errors['prediction'] == 2))
    | ((selected_errors['label'] == 2) & (selected_errors['prediction'] == 0))
)
recyclable_organic_errors = selected_errors.loc[recyclable_organic_mask].sort_values(
    ['confidence', 'top_two_margin'], ascending=False
)
write_csv_atomic(recyclable_organic_errors, ARTIFACT_DIR / 'oof_recyclable_organic_errors_{}.csv'.format(SELECTED_CANDIDATE))
write_csv_atomic(
    selected_errors.loc[selected_errors['confidence'] >= 0.90].sort_values('confidence', ascending=False),
    ARTIFACT_DIR / 'oof_high_confidence_errors_{}.csv'.format(SELECTED_CANDIDATE),
)

ERROR_ANALYSIS = {
    'selected_candidate': SELECTED_CANDIDATE,
    'oof_total_errors': int(len(selected_errors)),
    'oof_recyclable_organic_errors': int(len(recyclable_organic_errors)),
    'recyclable_organic_share_of_errors': (
        float(len(recyclable_organic_errors) / len(selected_errors)) if len(selected_errors) else 0.0
    ),
    'oof_electronic_errors': OOF_METRICS[SELECTED_CANDIDATE]['electronic_errors'],
    'oof_high_confidence_errors': int((selected_errors['confidence'] >= 0.90).sum()),
    'historical_v7_five_fold_oof_errors': V7_REFERENCE['five_fold_oof_total_errors'],
    'historical_v7_five_fold_oof_ro_errors': V7_REFERENCE['five_fold_oof_recyclable_organic_errors'],
    'comparability_note': (
        'V7 numbers come from the Transformers backend and the CLS + patch-mean cache; they are project '
        'context for the same folds and labels, not a protocol-matched causal comparison.'
    ),
    'test_images_inspected_for_model_decisions': False,
}
write_json_atomic(ERROR_ANALYSIS, ARTIFACT_DIR / 'error_analysis.json')
print(json.dumps(ERROR_ANALYSIS, indent=2))
display(recyclable_organic_errors.head(20))

## 26. Manifest reproduksibilitas, pemaketan artefak, dan ringkasan run

`reproducibility_manifest.json` menggabungkan seluruh tanda tangan menjadi satu berkas yang dapat diaudit: kontrak terkunci, environment, GPU, manifest model + commit SHA, kontrak data dan fold, tanda tangan preflight, tanda tangan cache train dan test, resep terkunci, keputusan seleksi, metadata full-training, dan manifest inferensi test. Ini yang perlu dilampirkan bila panitia meminta bukti bahwa prediksi tidak dibuat manual.

`artifact_bundle.zip` memuat seluruh bukti dan artefak reproduksibilitas tetapi **tidak** memuat cache fitur multi-gigabyte, kecuali `ZIP_INCLUDE_FEATURE_CACHE` diaktifkan.


In [ ]:
REPRODUCIBILITY_MANIFEST = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'experiment_purpose': EXPERIMENT_PURPOSE,
    'team_name': TEAM_NAME,
    'locked_contract': LOCKED_CONTRACT,
    'locked_contract_signature': LOCKED_CONTRACT_SIGNATURE,
    'environment': ENVIRONMENT_INFO,
    'gpus': GPU_RUNTIME_INFO,
    'device_map_devices': sorted(set(DEVICE_MAP.values())),
    'model_manifest': MODEL_MANIFEST,
    'data_contract_signature': DATA_CONTRACT_SIGNATURE,
    'fold_signature': FOLD_SIGNATURE,
    'preprocess_signature': PREPROCESS_SIGNATURE,
    'preflight_signature': PREFLIGHT_SIGNATURE,
    'preflight_passed': PREFLIGHT_REPORT['pass'],
    'train_cache_signature': TRAIN_CACHE_SIGNATURE,
    'test_cache_signature': TEST_CACHE_SIGNATURE,
    'recipe_signature': RECIPE_SIGNATURE,
    'selection_decision_signature': SELECTION_DECISION_SIGNATURE,
    'selected_candidate': SELECTED_CANDIDATE,
    'final_training_metadata': FINAL_TRAINING_METADATA,
    'test_inference_manifest': TEST_INFERENCE_MANIFEST,
    'competition_compliance': {
        'official_images_only': True,
        'official_raw_labels_are_canonical': True,
        'external_labeled_images_used': False,
        'metadata_used_as_predictive_feature': False,
        'test_used_for_training_validation_selection_or_tuning': False,
        'pseudo_labeling_on_test': False,
        'test_images_manually_inspected_for_model_decisions': False,
        'external_test_labels_loaded': False,
        'previous_submissions_loaded': False,
        'leaderboard_feedback_used': False,
        'test_accessed_only_after_selection_lock': True,
        'official_submission_row_order_preserved': True,
        'output_class_codes': [0, 1, 2],
        'public_pretrained_backbone_documented': ACTIVE_MODEL_REPO,
        'backbone_frozen': True,
    },
    'notebook_elapsed_seconds': time.perf_counter() - NOTEBOOK_STARTED_AT,
}
write_json_atomic(REPRODUCIBILITY_MANIFEST, ARTIFACT_DIR / 'reproducibility_manifest.json')

BUNDLE_PATH = ARTIFACT_DIR / 'artifact_bundle.zip'
bundled_files = []
skipped_files = 0
temporary_bundle = BUNDLE_PATH.with_name(BUNDLE_PATH.name + '.tmp')
if temporary_bundle.exists():
    temporary_bundle.unlink()
with zipfile.ZipFile(temporary_bundle, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(ARTIFACT_DIR.rglob('*')):
        if not path.is_file():
            continue
        relative = path.relative_to(ARTIFACT_DIR)
        if path.name.endswith('.tmp') or path == BUNDLE_PATH or path == temporary_bundle:
            skipped_files += 1
            continue
        if not ZIP_INCLUDE_FEATURE_CACHE and relative.parts and relative.parts[0] == 'feature_cache':
            skipped_files += 1
            continue
        archive.write(path, relative.as_posix())
        bundled_files.append(relative.as_posix())
os.replace(temporary_bundle, BUNDLE_PATH)

FINAL_RUN_SUMMARY = {
    'schema_version': 1,
    'revision_marker': REVISION_MARKER,
    'experiment_name': EXPERIMENT_NAME,
    'hypothesis': (
        'Patch-aware spatial and distributional pooling of frozen DINOv3 ViT-7B tokens carries signal that the '
        'V7 CLS + patch-mean cache discarded permanently.'
    ),
    'selected_candidate': SELECTED_CANDIDATE,
    'selection_reason': SELECTION_REASON,
    'rich_passed_development_gate': RICH_PASSED_DEVELOPMENT_GATE,
    'rich_adopted': RICH_ADOPTED,
    'development_gate_checks': DEVELOPMENT_GATE_CHECKS,
    'adoption_gate_checks': ADOPTION_GATE_CHECKS,
    'development_metrics': {
        candidate_name: {
            key: DEVELOPMENT_METRICS[candidate_name][key]
            for key in ('macro_f1', 'accuracy', 'log_loss', 'total_errors', 'ro_errors', 'electronic_errors', 'best_epoch')
        }
        for candidate_name in CANDIDATE_NAMES
    },
    'assessment_aggregate': {
        candidate_name: {
            key: ASSESSMENT_AGGREGATE[candidate_name][key]
            for key in ('macro_f1', 'accuracy', 'log_loss', 'total_errors', 'ro_errors', 'electronic_errors')
        }
        for candidate_name in ASSESSED_CANDIDATES
    },
    'five_fold_oof': {
        candidate_name: {
            key: OOF_METRICS[candidate_name][key]
            for key in ('macro_f1', 'accuracy', 'log_loss', 'total_errors', 'ro_errors', 'electronic_errors')
        }
        for candidate_name in ASSESSED_CANDIDATES
    },
    'historical_references': {'v7': V7_REFERENCE, 'v9a': V9A_REFERENCE},
    'full_training_epochs': FULL_TRAIN_EPOCHS,
    'cross_validation_best_epochs': selected_best_epochs,
    'submission_path': str(SUBMISSION_PATH),
    'submission_validation': SUBMISSION_VALIDATION,
    'predicted_class_distribution': TEST_INFERENCE_MANIFEST['predicted_class_distribution'],
    'timing_seconds': {
        'model_download': MODEL_DOWNLOAD_SECONDS,
        'model_load': MODEL_LOAD_SECONDS,
        'train_feature_cache_session': TRAIN_CACHE_MANIFEST['elapsed_seconds_this_session'],
        'test_feature_cache_session': TEST_CACHE_MANIFEST['elapsed_seconds_this_session'],
        'head_training_total': sum(
            metrics['training_seconds'] for metrics in list(DEVELOPMENT_METRICS.values()) + ASSESSMENT_RUNS
        ),
        'full_training': FINAL_TRAINING_METADATA['training_seconds'],
        'test_inference': TEST_INFERENCE_SECONDS,
        'notebook_total': time.perf_counter() - NOTEBOOK_STARTED_AT,
    },
    'throughput_images_per_second': {
        'preflight': PREFLIGHT_REPORT['stages']['D_throughput']['images_per_second'],
        'train_extraction': TRAIN_CACHE_MANIFEST['new_extraction_images_per_second'],
        'test_extraction': TEST_CACHE_MANIFEST['new_extraction_images_per_second'],
    },
    'artifact_bundle': {
        'path': str(BUNDLE_PATH),
        'file_count': len(bundled_files),
        'skipped_file_count': skipped_files,
        'feature_cache_included': ZIP_INCLUDE_FEATURE_CACHE,
        'size_bytes': BUNDLE_PATH.stat().st_size,
    },
    'test_images_accessed': TEST_ACCESS_STATE['test_images_accessed'],
    'manual_test_labels_accessed': TEST_ACCESS_STATE['manual_test_labels_accessed'],
}
write_json_atomic(FINAL_RUN_SUMMARY, ARTIFACT_DIR / 'final_run_summary.json')

print('Reproducibility manifest and artifact bundle written.')
print('Bundle: {} ({:.2f} MiB, {} files, feature cache included: {})'.format(
    BUNDLE_PATH,
    FINAL_RUN_SUMMARY['artifact_bundle']['size_bytes'] / 2 ** 20,
    len(bundled_files),
    ZIP_INCLUDE_FEATURE_CACHE,
))
print(json.dumps({
    'selected_candidate': SELECTED_CANDIDATE,
    'rich_adopted': RICH_ADOPTED,
    'five_fold_oof': FINAL_RUN_SUMMARY['five_fold_oof'],
    'full_training_epochs': FULL_TRAIN_EPOCHS,
    'submission_path': str(SUBMISSION_PATH),
    'notebook_total_seconds': FINAL_RUN_SUMMARY['timing_seconds']['notebook_total'],
}, indent=2))

## 27. Pintasan inferensi dari checkpoint tersimpan (verifikasi untuk juri)

Sel terakhir membuktikan bahwa submission dapat direproduksi **hanya** dari dua artefak: `final_head.pt` dan cache fitur test yang tersimpan. Head dibangun ulang dari arsitektur yang dicatat di metadata checkpoint, bobotnya dimuat dari disk, shard fitur test dimuat ulang dan divalidasi ulang dari disk (tanpa menjalankan backbone lagi, karena prapemrosesan dan definisi cabangnya terkunci oleh tanda tangan cache), lalu satu forward pass dijalankan.

Probabilitas yang dihasilkan harus cocok dengan `test_probabilities.npy` dan barisan prediksinya harus identik dengan submission yang sudah ditulis. Bila tidak, sel gagal. Ini adalah pemeriksaan otomatis; tidak ada prediksi yang diubah tangan.


In [ ]:
CHECKPOINT_REPRODUCTION_ATOL = 1e-6

shortcut_checkpoint = torch.load(FINAL_HEAD_PATH, map_location='cpu', weights_only=False)
shortcut_metadata = shortcut_checkpoint['metadata']
shortcut_head = build_head(shortcut_metadata['candidate']).to(HEAD_DEVICE)
shortcut_head.load_state_dict(shortcut_checkpoint['state_dict'], strict=True)
shortcut_head.eval()

# Reload the test features from the persisted shards; identical preprocessing is guaranteed by the cache signature.
shortcut_features, shortcut_manifest = build_feature_cache(
    'test', test_df, TEST_CACHE_DIR, TEST_CACHE_MANIFEST_PATH
)
if shortcut_manifest['cache_signature'] != TEST_CACHE_SIGNATURE:
    raise RuntimeError('The reloaded test cache signature differs from the one used for the submission.')

shortcut_probabilities = predict_probabilities(shortcut_head, shortcut_features, np.arange(len(test_df), dtype=np.int64))
stored_probabilities = np.load(ARTIFACT_DIR / 'test_probabilities.npy').astype(np.float64)
shortcut_predictions = shortcut_probabilities.argmax(axis=1).astype(np.int64)
stored_submission = pd.read_csv(SUBMISSION_PATH)

CHECKPOINT_REPRODUCTION = {
    'checkpoint': str(FINAL_HEAD_PATH),
    'candidate': shortcut_metadata['candidate'],
    'architecture': shortcut_metadata['candidate_spec']['architecture'],
    'branch_names_match': bool(shortcut_metadata['branch_names'] == list(BRANCH_NAMES)),
    'test_cache_signature_matches': True,
    'max_abs_probability_difference': float(np.abs(shortcut_probabilities - stored_probabilities).max()),
    'tolerance': CHECKPOINT_REPRODUCTION_ATOL,
    'predictions_identical': bool((shortcut_predictions == stored_submission['predicted'].to_numpy()).all()),
    'submission_row_order_identical': bool(
        stored_submission['id'].tolist() == submission_template['id'].tolist()
    ),
    'rows': int(len(stored_submission)),
}
CHECKPOINT_REPRODUCTION['pass'] = bool(
    CHECKPOINT_REPRODUCTION['branch_names_match']
    and CHECKPOINT_REPRODUCTION['max_abs_probability_difference'] <= CHECKPOINT_REPRODUCTION_ATOL
    and CHECKPOINT_REPRODUCTION['predictions_identical']
    and CHECKPOINT_REPRODUCTION['submission_row_order_identical']
    and CHECKPOINT_REPRODUCTION['rows'] == EXPECTED_TEST_ROWS
)
write_json_atomic(CHECKPOINT_REPRODUCTION, ARTIFACT_DIR / 'checkpoint_reproduction_check.json')
if not CHECKPOINT_REPRODUCTION['pass']:
    raise RuntimeError(
        'The saved checkpoint did not reproduce the submission: {}'.format(json.dumps(CHECKPOINT_REPRODUCTION, indent=2))
    )

del shortcut_checkpoint, shortcut_head, shortcut_features
release_cuda_cache()

print(json.dumps(CHECKPOINT_REPRODUCTION, indent=2))
print('\nV10 run complete.')
print('  selected candidate : {} ({})'.format(SELECTED_CANDIDATE, SELECTION_REASON))
print('  submission         : {}'.format(SUBMISSION_PATH))
print('  artifact bundle    : {}'.format(BUNDLE_PATH))
print('  total runtime      : {:.1f} minutes'.format((time.perf_counter() - NOTEBOOK_STARTED_AT) / 60.0))